# 03d — Flex: blend the fine-tune's weights with base (WiSE-FT)

Step 3 of the protocol (D105). Decode only. Every weight is set to
`(1 - alpha) * base + alpha * fine-tuned`, the same alpha for every tensor, and the blend is scored
like any other model: gold and val, then the public sets.

**Why.** Fine-tuning erases vocabulary our train labels never contain, which costs about a point on
the public sets (03b). Averaging back towards base recovers it: on 2026-09-27, alpha = 0.5 kept our
domain (val +0.05, gold -0.32 against the fine-tune) and brought every public set back to base
(findings.md, *Weight blending recovers the loss*).

**The rule, fixed on 2026-09-27 before any blend was scored** (`sweep.choose_blend`): choose on
val, taking the blend closest to base whose val WER is within 0.3 points of the fine-tuned
model's. Gold and the public sets are reported, never used to choose. If no blend qualifies, the
fine-tuned model is kept.

**What is blended.** `SOURCE` names the fine-tuned run; left at `None` it is the winner in 03c's
`ablation.json`. To skip the ablation, set `SOURCE = "vanilla-s1"`.

**No weights are written here.** A blend is rebuilt in seconds from the two models; 03e rebuilds
the chosen one and writes it once.

**Needs** 03b's `base` run (base Flex on gold, val and the public sets) for the pairing, and the
source run's weights in `OUT_REPO`.

**Outputs**: `OUT_REPO/<RUN_PREFIX>/blend-<alpha>/` per blend (scores and transcripts) and
`OUT_REPO/<RUN_PREFIX>/blend.json`, which names the choice 03e freezes.

**Smoke run first.** `SMOKE = True` in Config trains and scores on a few hundred clips for one
epoch and writes under `<RUN_PREFIX>-smoke`, so every cell runs end to end in minutes before a
real run is paid for. Nothing here has been verified on a GPU until its smoke run has passed.

## Config

In [ ]:
NOTEBOOK = "03d_Flex_Blend"
RUN_PREFIX = "flex-2026-09-30"   # OUT_REPO/<RUN_PREFIX>/<run>/: one folder per run, shared by 03a-03e
DATASET_EXPORT = "2026-09-30"    # the export every run trains and is scored on; another is refused
MODEL_ID = "bodhan-ai/indic-transcribe-flex"
LANG, MODE = "ne", "mixed"
OUT_REPO = "Sagyam/nepanglish-asr-flex-ft"  # private HF model repo
MODEL_NAME = "Indic-Transcribe-Flex FT"     # shown on the harness's Models page (D83)
SMOKE = False                    # True: a few hundred clips, one epoch, under <RUN_PREFIX>-smoke
PAD_TO_S = 1.0
EVAL_BUDGET_S, EVAL_ITEMS = 1200.0, 96
DATA_LOCAL = None                # a local export in the HF layout instead of the download
SOURCE = None               # the fine-tuned run to blend; None reads the winner from 03c's ablation.json
ALPHAS = (0.25, 0.5, 0.75)  # the grid, fixed in advance
TOLERANCE = 0.3             # points of val WER a blend may cost against the fine-tune

## Setup

In [ ]:
%pip install -q rapidfuzz duckdb pyarrow
import json
import os
import sys
from pathlib import Path

# Before torch touches CUDA: lets the allocator grow segments instead of fragmenting when batch
# shapes vary (a fresh kernel only).
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

IN_COLAB = "google.colab" in sys.modules
# Secrets only work from a cell run in the Colab UI. When cells are driven from outside (the Colab
# MCP), run this cell by hand once. The token is then kept in the hub's own token file on the VM
# (where `hf auth login` puts it), so a kernel restart needs no click; the file goes when the
# runtime is deleted.
if IN_COLAB:
    from google.colab import userdata

    TOKEN_FILE = Path.home() / ".cache" / "huggingface" / "token"
    if not os.environ.get("HF_TOKEN"):
        os.environ["HF_TOKEN"] = (TOKEN_FILE.read_text().strip() if TOKEN_FILE.exists()
                                  else userdata.get("HF_TOKEN"))
    if not TOKEN_FILE.exists():
        TOKEN_FILE.parent.mkdir(parents=True, exist_ok=True)
        TOKEN_FILE.write_text(os.environ["HF_TOKEN"])
        TOKEN_FILE.chmod(0o600)
FT = Path("/content/ft") if IN_COLAB else Path.cwd() / ".cache-ft"
FT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(FT))
if SMOKE:
    RUN_PREFIX += "-smoke"  # a smoke run's outputs never land among the real runs
OUT_ROOT = FT / "out" / RUN_PREFIX
OUT_ROOT.mkdir(parents=True, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print("runs:", RUN_PREFIX, "| outputs:", OUT_ROOT, "| SMOKE RUN" if SMOKE else "")

In [ ]:
%%writefile /content/ft/ftkit.py
"""Shared fine-tuning kit for the Nepanglish ASR notebooks (Flex, 03a-03e; the students, 06).

Everything that is not model-specific: the dataset held in RAM, duration-bucketed batches, the
harness scorer, a GPU utilisation monitor, a batch-size probe and one training loop. Each notebook
writes this file out, so it imports exactly the same code as the main kernel. Keep it importable
on Python 3.10+ with numpy 1.x or 2.x.

How the GPU is kept busy:
  * audio is decoded once into RAM as int16; a clip is a slice, so no disk I/O in the loop;
  * batches are built by duration and padded to whole seconds, so there is little padding and
    only ~20 distinct shapes (cudnn.benchmark can then pick kernels once per shape);
  * DataLoader workers collate and pin the next batches while the GPU runs the current one;
  * a probe finds the largest micro-batch that survives forward+backward at the longest clip,
    with the optimizer state already allocated, and training uses a fixed fraction of it;
  * bf16 autocast, TF32 matmuls and fused AdamW; gradient accumulation reaches the effective
    batch; utilisation, throughput and padding waste are logged, not assumed.
"""

from __future__ import annotations

import glob
import json
import math
import random
import re
import shutil
import subprocess
import sys
import threading
import time
from collections import Counter
from collections.abc import Callable, Iterator, Sequence
from contextlib import contextmanager
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import numpy as np
import soundfile as sf
import torch

REPO = "Sagyam/nepanglish-asr"
SR = 16_000


# --- data --------------------------------------------------------------------------------------


def download_dataset(
    local: str | None = None, episodes: Sequence[str] = (), analytics: bool = False
) -> Path:
    """The HF dataset snapshot, or a local copy in the same layout.

    Labels, manifests and the harness's scorer always come; audio only for the recordings named
    in `episodes` (whole FLACs, 8 GiB for every split), and the analytics export, which carries
    the speaker turns, only when asked. Call it once without episodes to read the splits, then
    again with the episodes the notebook decodes or trains on: both calls return the same folder.
    """
    if local:
        return Path(local)
    from huggingface_hub import snapshot_download

    patterns = ["training/training.jsonl", "training/manifest.json", "gold/*", "harness/*"]
    if analytics:
        patterns.append("analytics/*")
    patterns += [f"training/episodes/{glob.escape(e)}.flac" for e in sorted(set(episodes))]
    return Path(snapshot_download(REPO, repo_type="dataset", allow_patterns=patterns))


def load_splits(data: Path) -> dict[str, list[dict]]:
    """train / val from the training export, gold from the gold export; asserts disjointness."""
    rows = [
        json.loads(line) for line in (data / "training" / "training.jsonl").open(encoding="utf-8")
    ]
    gold = [json.loads(line) for line in (data / "gold" / "gold.jsonl").open(encoding="utf-8")]
    splits = {
        "train": [r for r in rows if r["split"] == "train"],
        "val": [r for r in rows if r["split"] == "val"],
        "gold": gold,
    }
    trained = {r["segment_id"] for r in rows}
    assert not trained & {r["segment_id"] for r in gold}, "a gold clip is in the training export"
    return splits


def attach_speaker_turns(data: Path, rows: Sequence[dict]) -> int:
    """Copy each row's diarized turns, with their linked voices (D78, D87), from the analytics
    export: the training export does not carry them. Returns how many rows have turns."""
    want = {r["segment_id"]: r for r in rows}
    with (data / "analytics" / "analytics.jsonl").open(encoding="utf-8") as fh:
        for line in fh:
            a = json.loads(line)
            if a["segment_id"] in want:
                want[a["segment_id"]]["speaker_turns"] = a.get("speaker_turns")
    return sum(bool(r.get("speaker_turns")) for r in rows)


def duration(row: dict) -> float:
    return row["end_time"] - row["start_time"]


class AudioStore:
    """Every episode decoded once into RAM as int16; a clip is a slice of it (a view, no copy).

    ``folder`` is where the whole recordings are, under ``data``: the labelled export's
    ``training/episodes`` by default, or the distillation corpus's ``distill/episodes`` (D101)."""

    def __init__(self, data: Path, episode_ids: Sequence[str], folder: str = "training/episodes"):
        self.audio: dict[str, np.ndarray] = {}
        for ep in sorted(set(episode_ids)):
            audio, sr = sf.read(data / folder / f"{ep}.flac", dtype="int16")
            assert sr == SR and audio.ndim == 1, (ep, sr, audio.shape)
            self.audio[ep] = audio

    def clip(self, row: dict) -> np.ndarray:
        if "audio" in row:  # a public-benchmark clip carries its own audio (evalkit)
            return row["audio"]
        audio = self.audio[row["episode_id"]]
        return audio[round(row["start_time"] * SR) : round(row["end_time"] * SR)]

    def clip_f32(self, row: dict) -> torch.Tensor:
        return torch.from_numpy(self.clip(row).astype(np.float32) / 32768.0)

    @property
    def gib(self) -> float:
        return sum(a.nbytes for a in self.audio.values()) / 2**30


def bucket_batches(
    rows: Sequence[dict],
    *,
    budget_s: float,
    max_items: int,
    pad_to_s: float,
    shuffle: bool,
    seed: int = 0,
) -> list[list[int]]:
    """Index batches whose padded size (items x longest clip, rounded up to `pad_to_s`) fits
    `budget_s`. Sorting by duration keeps padding low; shuffling reorders whole batches and
    jitters lengths slightly so the same clips do not always share a batch."""
    rng = random.Random(seed)
    jitter = [rng.uniform(-0.3, 0.3) if shuffle else 0.0 for _ in rows]
    order = sorted(range(len(rows)), key=lambda i: duration(rows[i]) + jitter[i])
    batches: list[list[int]] = []
    cur: list[int] = []
    longest = 0.0
    for i in order:
        d = math.ceil(duration(rows[i]) / pad_to_s) * pad_to_s
        if cur and ((len(cur) + 1) * max(longest, d) > budget_s or len(cur) >= max_items):
            batches.append(cur)
            cur, longest = [], 0.0
        cur.append(i)
        longest = max(longest, d)
    if cur:
        batches.append(cur)
    if shuffle:
        rng.shuffle(batches)
    return batches


def pad_len(samples: int, pad_to_s: float) -> int:
    step = int(pad_to_s * SR)
    return math.ceil(samples / step) * step


class _Batches(torch.utils.data.Dataset):
    def __init__(self, rows, batches, collate):
        self.rows, self.batches, self.collate = rows, batches, collate

    def __len__(self):
        return len(self.batches)

    def __getitem__(self, i):
        return self.collate([self.rows[j] for j in self.batches[i]])


def loader(rows, batches, collate, workers: int) -> torch.utils.data.DataLoader:
    return torch.utils.data.DataLoader(
        _Batches(rows, batches, collate),
        batch_size=None,
        shuffle=False,
        num_workers=workers,
        pin_memory=True,
        prefetch_factor=4 if workers else None,
        persistent_workers=False,
    )


# --- scoring -----------------------------------------------------------------------------------


def is_loop(text: str) -> bool:
    """A 3-word sequence repeated 5 or more times: the decoder is stuck, not transcribing."""
    toks = text.split()
    top = Counter(zip(toks, toks[1:], toks[2:], strict=False)).most_common(1)
    return bool(top) and top[0][1] >= 5


class RetryLoops:
    """Wraps a batch `transcribe`: a clip whose first decode loops is decoded again, alone, by
    `retry` (anti-repetition settings); every other clip keeps its first decode untouched.
    `log` keeps (segment_id, first, retried), so the effect is measurable within one run."""

    def __init__(self, transcribe: Callable[[list[dict]], list[str]], retry: Callable[[dict], str]):
        self.transcribe, self.retry = transcribe, retry
        self.log: list[tuple[str, str, str]] = []

    def __call__(self, rows: list[dict]) -> list[str]:
        texts = self.transcribe(rows)
        for i, text in enumerate(texts):
            if is_loop(text):
                texts[i] = self.retry(rows[i])
                self.log.append((rows[i]["segment_id"], text, texts[i]))
        return texts


def harness_scorer(data: Path, work: Path) -> Callable[[Sequence[str], Sequence[str]], dict]:
    """fold.py from the dataset's harness/, imported from a real copy: the HF cache stores files as
    symlinks into its blob store, and normalize.py finds its config via its resolved path.

    Two layouts are accepted: the repo's (backend/app/services/, config/), and the flat one the
    2026-09-21 export uploaded (fold.py, normalize.py, normalization.yaml side by side), which
    is laid back out here because normalize.py looks for ../../../config/. error_mining.py,
    error_store.py and attribution.py travel with fold.py when the copy has them
    (docs/WER-Breakdown.md); evalkit writes no error files without them.

    Each `per_clip` count keeps the clip's folded `alignment`, so mining its errors aligns nothing
    again; `summarize` adds up everything else."""
    src, dst = data / "harness", work / "harness"
    if not dst.exists():
        if (src / "fold.py").exists():
            (dst / "backend" / "app" / "services").mkdir(parents=True)
            (dst / "config").mkdir()
            for name in (
                "fold.py",
                "normalize.py",
                "error_mining.py",
                "error_store.py",
                "attribution.py",
            ):
                if name in ("fold.py", "normalize.py") or (src / name).exists():
                    shutil.copyfile(src / name, dst / "backend" / "app" / "services" / name)
            shutil.copyfile(src / "normalization.yaml", dst / "config" / "normalization.yaml")
        else:
            shutil.copytree(src, dst)
    for name in [m for m in sys.modules if m == "app" or m.startswith("app.")]:
        del sys.modules[name]
    sys.path.insert(0, str(dst / "backend"))
    from app.services.fold import fold_tokens, fold_version, word_errors
    from rapidfuzz.distance import Levenshtein

    dev = re.compile(r"[ऀ-ॿ]")

    def chars(text: str) -> str:
        return " ".join(t if dev.search(t) else t.lower() for t in fold_tokens(text))

    def per_clip(refs: Sequence[str], hyps: Sequence[str]) -> list[dict]:
        """Every count a score needs, per clip, from one alignment each: `summarize` adds any
        subset of them up, so scoring per class or pairing against another system aligns
        nothing again. errors = sub + del + ins (folded)."""
        out = []
        for ref, hyp in zip(refs, hyps, strict=True):
            folded = word_errors(ref, hyp)
            raw = word_errors(ref, hyp, folded=False)
            ref_chars, hyp_chars = chars(ref), chars(hyp)
            out.append(
                {
                    "words": len(fold_tokens(ref)),
                    "errors": folded.errors,
                    "sub": folded.substitutions,
                    "del": folded.deletions,
                    "ins": folded.insertions,
                    "raw_words": raw.ref_words,
                    "raw_errors": raw.errors,
                    "chars": len(ref_chars),
                    "char_errors": Levenshtein.distance(ref_chars, hyp_chars),
                    "loop": int(is_loop(hyp)),
                    "alignment": folded,
                }
            )
        return out

    def summarize(clips: Sequence[dict]) -> dict:
        """The score of a set of clips from their `per_clip` counts."""
        t = (
            {k: sum(c[k] for c in clips) for k in clips[0] if k != "alignment"}
            if clips
            else Counter()
        )
        words = max(t["words"], 1)
        folded = t["sub"] + t["del"] + t["ins"]
        assert folded == t["errors"], "S + D + I must add up to the folded errors"
        return {
            "wer": 100 * t["errors"] / words,
            "raw_wer": 100 * t["raw_errors"] / max(t["raw_words"], 1),
            "cer": 100 * t["char_errors"] / max(t["chars"], 1),
            # folded, per 100 reference words. In crosstalk a label keeps what was audible, which is
            # not always both voices, so a model that writes the other one can be charged insertions
            "sub": 100 * t["sub"] / words,
            "del": 100 * t["del"] / words,
            "ins": 100 * t["ins"] / words,
            "loops": t["loop"],
            "clips": len(clips),
        }

    def score(refs: Sequence[str], hyps: Sequence[str]) -> dict:
        return summarize(per_clip(refs, hyps))

    score.per_clip = per_clip
    score.summarize = summarize
    score.fold_version = fold_version()  # for the model card: which fold the numbers are in
    return score


def transcribe_rows(
    rows: Sequence[dict],
    transcribe: Callable[[list[dict]], list[str]],
    *,
    budget_s: float,
    max_items: int,
    pad_to_s: float,
) -> tuple[list[str], list[float]]:
    """Run `transcribe` over duration-bucketed batches; returns texts and per-clip compute seconds
    in the original row order."""
    texts: list[str] = [""] * len(rows)
    compute = [0.0] * len(rows)
    for batch in bucket_batches(
        rows, budget_s=budget_s, max_items=max_items, pad_to_s=pad_to_s, shuffle=False
    ):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        out = transcribe([rows[i] for i in batch])
        torch.cuda.synchronize()
        took = time.perf_counter() - t0
        audio_s = sum(duration(rows[i]) for i in batch)
        for i, text in zip(batch, out, strict=True):
            texts[i] = text.strip()
            compute[i] = took * duration(rows[i]) / audio_s
    return texts, compute


def write_hyps(
    path: Path, rows: Sequence[dict], texts: Sequence[str], compute: Sequence[float]
) -> None:
    """The hypothesis-cache format the bake-off comparisons used, so the result can join them."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as fh:
        for r, text, c in zip(rows, texts, compute, strict=True):
            fh.write(
                json.dumps(
                    {"segment_id": r["segment_id"], "text": text, "compute_s": c},
                    ensure_ascii=False,
                )
                + "\n"
            )


# --- GPU ---------------------------------------------------------------------------------------


def fast_cuda() -> None:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True  # safe: shapes are padded to whole seconds
    torch.set_float32_matmul_precision("high")


class GpuMonitor:
    """Samples nvidia-smi in a background thread; `window()` returns the mean utilisation and the
    peak memory since the previous call."""

    def __init__(self, every: float = 1.0):
        self.every, self._util, self._mem = every, [], []
        self._stop = threading.Event()
        self._thread = threading.Thread(target=self._run, daemon=True)

    def _run(self):
        while not self._stop.is_set():
            try:
                out = subprocess.run(
                    [
                        "nvidia-smi",
                        "--query-gpu=utilization.gpu,memory.used",
                        "--format=csv,noheader,nounits",
                    ],
                    capture_output=True,
                    text=True,
                    timeout=5,
                )
                util, mem = (float(x) for x in out.stdout.strip().splitlines()[0].split(","))
                self._util.append(util)
                self._mem.append(mem)
            except Exception:
                pass
            self._stop.wait(self.every)

    def start(self) -> GpuMonitor:
        self._thread.start()
        return self

    def stop(self) -> None:
        self._stop.set()

    def window(self) -> dict:
        util, mem = self._util, self._mem
        self._util, self._mem = [], []
        return {
            "gpu_util": float(np.mean(util)) if util else float("nan"),
            "gpu_mem_gib": max(mem) / 1024 if mem else float("nan"),
        }


def init_optimizer_state(model: torch.nn.Module, optimizer: torch.optim.Optimizer) -> None:
    """Allocate AdamW's moment buffers before probing, without moving any weight: with zero
    gradients and zero weight decay an AdamW step is a no-op, but it creates the state."""
    decay = [g["weight_decay"] for g in optimizer.param_groups]
    for g in optimizer.param_groups:
        g["weight_decay"] = 0.0
    for p in model.parameters():
        if p.requires_grad:
            p.grad = torch.zeros_like(p)
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)
    for g, d in zip(optimizer.param_groups, decay, strict=True):
        g["weight_decay"] = d
    for state in optimizer.state.values():
        if "step" in state:
            state["step"].zero_()


@contextmanager
def batchnorm_kept(model: torch.nn.Module) -> Iterator[None]:
    """Restore every BatchNorm layer's running statistics on exit. A forward pass in train mode
    updates them with no optimizer step at all: on 2026-10-01 the probe's random noise had
    rewritten about half of Flex's, and 03a's "val WER before training" read 13.81 for a model
    that scores 9.24."""
    bns = [m for m in model.modules() if isinstance(m, torch.nn.modules.batchnorm._BatchNorm)]
    saved = [{k: v.clone() for k, v in m.state_dict().items()} for m in bns]
    try:
        yield
    finally:
        for m, s in zip(bns, saved, strict=True):
            m.load_state_dict(s)


def probe_max_items(step: Callable[[int], None], lo: int, hi: int, model: torch.nn.Module) -> int:
    """The largest n in [lo, hi] for which `step(n)` (one forward+backward at the worst case)
    fits in memory. `step` must not touch the gradients. The model's weights never move, and its
    BatchNorm statistics are restored after, so it leaves the model as it found it.

    The gradient buffer stays allocated throughout, as it does in training from the second
    micro-batch of every accumulated step: freeing it between probes measured the activations
    without it and picked a batch that ran out of memory once training accumulated."""
    params = [p for p in model.parameters() if p.requires_grad]
    for p in params:
        p.grad = torch.zeros_like(p)
    best = 0
    with batchnorm_kept(model):
        while lo <= hi:
            mid = (lo + hi) // 2
            try:
                step(mid)
                torch.cuda.synchronize()
                best, lo = mid, mid + 1
            except torch.cuda.OutOfMemoryError:
                hi = mid - 1
            for p in params:
                p.grad.zero_()
            torch.cuda.empty_cache()
    for p in params:
        p.grad = None
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    return best


# --- training ----------------------------------------------------------------------------------


@dataclass
class TrainConfig:
    name: str
    out: str
    epochs: int = 10
    lr: float = 1e-5
    schedule: str = "linear"  # linear warmup+decay, or "tristage" (fairseq2's default)
    warmup_frac: float = 0.1
    effective_s: float = 720.0  # audio seconds per optimizer step, reached by accumulation
    weight_decay: float = 0.0
    clip_norm: float = 1.0
    patience: int = 3  # evaluations without a val-WER improvement before stopping
    # WER points an evaluation must gain on the last counted one to reset `patience`. 0 counts
    # any gain, so a crawl of 0.2 points an epoch never stops. The best weights are kept either way.
    min_delta: float = 0.0
    seed: int = 0
    log_every: int = 10
    workers: int = 6


def lr_factor(step: int, total: int, cfg: TrainConfig) -> float:
    warm = max(1, int(cfg.warmup_frac * total))
    if step < warm:
        return (step + 1) / warm
    if cfg.schedule == "tristage":  # 10% warmup, 40% hold, 50% exponential decay to 5%
        hold_end = int(0.5 * total)
        if step < hold_end:
            return 1.0
        frac = (step - hold_end) / max(1, total - hold_end)
        return math.exp(math.log(0.05) * frac)
    return max(0.0, (total - step) / max(1, total - warm))


def group_steps(rows, batches, effective_s: float) -> list[list[list[int]]]:
    """Micro-batches grouped into optimizer steps of about `effective_s` seconds of real audio."""
    steps, cur, acc = [], [], 0.0
    for b in batches:
        cur.append(b)
        acc += sum(duration(rows[i]) for i in b)
        if acc >= effective_s:
            steps.append(cur)
            cur, acc = [], 0.0
    if cur:
        steps.append(cur)
    return steps


@contextmanager
def timed(label: str) -> Iterator[None]:
    """Print `label...` before a step and `label: done in N s` after it, so a step that stalls
    shows which one it is instead of leaving a cell silent."""
    print(f"{label}...", flush=True)
    t0 = time.perf_counter()
    yield
    print(f"{label}: done in {time.perf_counter() - t0:.0f} s", flush=True)


def sid(m: dict) -> str:
    """A score's error split, per 100 reference words: S + D + I is its folded WER. Read all three
    on crosstalk: the labels drop the other voice, so hearing it costs insertions."""
    return f"S {m['sub']:.2f}  D {m['del']:.2f}  I {m['ins']:.2f}"


def retry_note(val: dict) -> str:
    """For an `evaluate` that wraps its decode in RetryLoops and reports the greedy score too."""
    if "retried" not in val:
        return ""
    return f"  (greedy WER {val['greedy_wer']:.2f}, {val['retried']} retried)"


def speed_check(
    model: torch.nn.Module,
    *,
    rows: Sequence[dict],
    batches: list[list[int]],
    collate: Callable[[list[dict]], Any],
    loss_fn: Callable[[torch.nn.Module, Any], tuple[torch.Tensor, int]],
    evaluate: Callable[[torch.nn.Module], dict],
    val_rows: Sequence[dict],
    gold_rows: Sequence[dict],
    epochs: int,
    monitor: GpuMonitor,
    workers: int = 6,
    n_micro: int = 10,
) -> dict:
    """Time the run before committing to it: forward+backward on `n_micro` real training
    micro-batches (no optimizer step, so no weight moves), then one full val pass, which is also
    the val WER before training. Projects the wall time of every epoch and the gold pass.

    The micro-batches run twice and only the second pass is timed, so cudnn's per-shape kernel
    search and worker start-up are excluded. Gradients stay allocated between micro-batches, as
    they do under accumulation, so the memory reading is training's. BatchNorm running
    statistics are restored after."""
    sample = batches[:n_micro]
    it = iter(loader(rows, sample + sample, collate, workers))

    def run() -> tuple[float, float, int]:
        audio = padded = 0.0
        clips = 0
        for _ in sample:
            b = next(it)
            with torch.autocast("cuda", dtype=torch.bfloat16):
                loss, _ = loss_fn(model, b)
            loss.backward()
            model.zero_grad(set_to_none=False)
            audio, padded = audio + b["seconds"], padded + b["padded_seconds"]
            clips += len(b["lens"]) if "lens" in b else b["wav"].shape[0]
        torch.cuda.synchronize()
        return audio, padded, clips

    model.train()
    with batchnorm_kept(model):
        run()
        monitor.window()
        t0 = time.perf_counter()
        audio, padded, clips = run()
        train_dt = time.perf_counter() - t0
        gpu = monitor.window()
        model.zero_grad(set_to_none=True)

    model.eval()
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        val = evaluate(model)
    torch.cuda.synchronize()
    val_dt = time.perf_counter() - t0
    model.train()

    secs = lambda rs: sum(duration(r) for r in rs)  # noqa: E731
    epoch_s = secs(rows) / (audio / train_dt)
    gold_s = val_dt * secs(gold_rows) / secs(val_rows)
    rec = {
        "train_x_realtime": audio / train_dt,
        "train_ms_per_clip": 1000 * train_dt / clips,
        "padding_waste": 1 - audio / padded,
        **gpu,
        "val_s": val_dt,
        "val_ms_per_clip": 1000 * val_dt / len(val_rows),
        **{f"val_before_{k}": v for k, v in val.items()},
        "epoch_min": epoch_s / 60,
        "projected_h": (epochs * (epoch_s + val_dt) + gold_s) / 3600,
    }
    print(
        f"train: {rec['train_x_realtime']:.0f}x realtime = "
        f"{rec['train_ms_per_clip']:.0f} ms per clip "
        f"(forward+backward, {clips} clips), pad waste {rec['padding_waste']:.0%}, "
        f"GPU {rec['gpu_util']:.0f}% util, {rec['gpu_mem_gib']:.1f} GiB\n"
        f"val:   {val_dt:.0f} s for {len(val_rows)} clips = "
        f"{rec['val_ms_per_clip']:.0f} ms per clip "
        f"(batched decode); WER before training {val['wer']:.2f} ({sid(val)}), loops {val['loops']}"
        f"{retry_note(val)}\n"
        f"projected: {rec['epoch_min']:.1f} min per epoch + {val_dt / 60:.1f} min val -> "
        f"at most {rec['projected_h']:.2f} h for {epochs} epochs and gold "
        "(early stopping can cut it)",
        flush=True,
    )
    return rec


class HubResume:
    """One run's resume point in a scratch model repo on the hub: `<path>/resume.pt`.

    `train` saves it after every epoch's evaluation and reads it when it starts, so a lost
    runtime costs the epoch in progress, not the run. It holds the weights as they are, the best
    weights when they differ, the counters and the history; not the optimizer's moments, which
    are twice the model's size and rebuild within a few steps. Every save replaces the last one,
    and the blobs it supersedes are deleted from the repo's storage, which is why this lives in a
    repo of its own: nothing else can be deleted by mistake. A failed save, load or clean-up is
    reported and never stops training."""

    NAME = "resume.pt"

    def __init__(self, api: Any, repo: str, path: str, local: Path):
        self.api, self.repo, self.path, self.local = api, repo, path, Path(local)

    @property
    def remote(self) -> str:
        return f"{self.path}/{self.NAME}"

    def load(self) -> dict | None:
        try:
            if not self.api.file_exists(self.repo, self.remote):
                return None
            from huggingface_hub import hf_hub_download

            with timed(f"resuming from {self.repo}/{self.remote}"):
                path = hf_hub_download(self.repo, self.remote, token=self.api.token)
                return torch.load(path, map_location="cpu", weights_only=True)
        except Exception as exc:  # a broken resume point means starting over, not failing
            print(
                f"no usable resume point ({type(exc).__name__}: {exc}); starting over", flush=True
            )
            return None

    def save(self, state: dict) -> None:
        try:
            self.local.mkdir(parents=True, exist_ok=True)
            torch.save(state, self.local / self.NAME)
            with timed(f"saving the resume point after epoch {state['epoch']}"):
                self.api.upload_file(
                    path_or_fileobj=str(self.local / self.NAME),
                    path_in_repo=self.remote,
                    repo_id=self.repo,
                    commit_message=f"{self.path}: after epoch {state['epoch']}",
                )
            self._purge(keep_newest=True)
        except Exception as exc:
            print(f"resume point not saved ({type(exc).__name__}: {exc})", flush=True)

    def clear(self) -> None:
        """Remove the resume point once the run's weights are safely uploaded elsewhere."""
        try:
            if self.api.file_exists(self.repo, self.remote):
                self.api.delete_file(
                    self.remote, self.repo, commit_message=f"{self.path}: run finished"
                )
            self._purge(keep_newest=False)
            shutil.rmtree(self.local, ignore_errors=True)
        except Exception as exc:
            print(f"resume point not cleared ({type(exc).__name__}: {exc})", flush=True)

    def _purge(self, keep_newest: bool) -> None:
        """Delete this run's superseded blobs for good: a replaced file stays in the repo's
        history, and its storage stays counted, until it is removed from LFS as well."""
        mine = sorted(
            (f for f in self.api.list_lfs_files(self.repo) if f.filename == self.remote),
            key=lambda f: f.pushed_at,
        )
        stale = mine[:-1] if keep_newest else mine
        if stale:
            self.api.permanently_delete_lfs_files(self.repo, stale)


def train(
    model: torch.nn.Module,
    *,
    cfg: TrainConfig,
    rows: Sequence[dict],
    make_batches: Callable[[int], list[list[int]]],
    collate: Callable[[list[dict]], Any],
    loss_fn: Callable[[torch.nn.Module, Any], tuple[torch.Tensor, int]],
    evaluate: Callable[[torch.nn.Module], dict],
    save_best: Callable[[torch.nn.Module], None],
    optimizer: torch.optim.Optimizer,
    monitor: GpuMonitor,
    resume: Any | None = None,
) -> dict:
    """Train with bf16 autocast and gradient accumulation; evaluate on val after every epoch;
    keep the best weights (by val WER) in CPU memory and hand them to `save_best`. A hand stop
    (KeyboardInterrupt) after the first evaluation ends training the same way.

    `loss_fn` returns a *summed* loss and the number of units it sums over (clips for CTC, tokens
    for cross-entropy); gradients are divided by the step's total units, so accumulated
    micro-batches of different sizes are weighted exactly. Count the units on the CPU (in
    `collate`): an `int()` of a GPU tensor stalls the host once per micro-batch.

    `resume` (a `HubResume`) makes the run restartable: its state is saved after every epoch and,
    when one is found at the start, training continues from the epoch after it, on the same
    batches and at the same point of the LR schedule. The optimizer's moments start again."""
    out = Path(cfg.out)
    out.mkdir(parents=True, exist_ok=True)
    torch.manual_seed(cfg.seed)
    plan = [group_steps(rows, make_batches(e), cfg.effective_s) for e in range(cfg.epochs)]
    total = sum(len(p) for p in plan)
    base_lrs = [g["lr"] for g in optimizer.param_groups]
    params = [p for p in model.parameters() if p.requires_grad]
    history, best, best_state, bad, step = [], float("inf"), None, 0, 0
    counted = float("inf")  # the val WER that last reset `bad`
    first_epoch, finished = 0, False
    state = resume.load() if resume is not None else None
    if state is not None:
        model.load_state_dict(state["model"])
        best_state = state["best_model"] or state["model"]
        history, best, counted = state["history"], state["best"], state["counted"]
        bad, step, first_epoch, finished = (
            state["bad"],
            state["step"],
            state["epoch"],
            state["done"],
        )
        print(
            f"{cfg.name}: resumed after epoch {first_epoch} (best val WER {best:.2f})", flush=True
        )
    print(
        f"{cfg.name}: {total} optimizer steps over {cfg.epochs} epochs "
        f"(~{cfg.effective_s / 60:.0f} min of audio each), peak lr {cfg.lr:g}, {cfg.schedule}"
    )
    stopped_by_hand = False
    try:
        for epoch in range(first_epoch, 0 if finished else cfg.epochs):
            model.train()
            flat = [b for s in plan[epoch] for b in s]
            sizes = [len(s) for s in plan[epoch]]
            it = iter(loader(rows, flat, collate, cfg.workers))
            t_log, audio_log, padded_log, loss_log, units_log = (
                time.perf_counter(),
                0.0,
                0.0,
                0.0,
                0,
            )
            monitor.window()
            for n_micro in sizes:
                for g, lr0 in zip(optimizer.param_groups, base_lrs, strict=True):
                    g["lr"] = lr0 * lr_factor(step, total, cfg)
                step_units = 0
                for _ in range(n_micro):
                    batch = next(it)
                    with torch.autocast("cuda", dtype=torch.bfloat16):
                        loss, units = loss_fn(model, batch)
                    loss.backward()
                    step_units += units
                    loss_log += loss.detach()  # stays on the GPU: no host sync per micro-batch
                    units_log += units
                    audio_log += batch["seconds"]
                    padded_log += batch["padded_seconds"]
                torch._foreach_div_(
                    [p.grad for p in params if p.grad is not None], max(step_units, 1)
                )
                gnorm = torch.nn.utils.clip_grad_norm_(params, cfg.clip_norm)
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                step += 1
                if step % cfg.log_every == 0:
                    torch.cuda.synchronize()
                    dt = time.perf_counter() - t_log
                    gpu = monitor.window()
                    rec = {
                        "step": step,
                        "epoch": epoch + 1,
                        "lr": optimizer.param_groups[0]["lr"],
                        "loss": float(loss_log) / max(units_log, 1),
                        "grad_norm": float(gnorm),
                        "audio_x_realtime": audio_log / dt,
                        "padding_waste": 1 - audio_log / max(padded_log, 1e-9),
                        "peak_alloc_gib": torch.cuda.max_memory_allocated() / 2**30,
                        **gpu,
                    }
                    history.append(rec)
                    print(
                        f"step {step:5d}/{total} ep {epoch + 1} lr {rec['lr']:.2e} "
                        f"loss {rec['loss']:.4f} | {rec['audio_x_realtime']:6.0f}x realtime, "
                        f"pad waste {rec['padding_waste']:.0%}, "
                        f"GPU {rec['gpu_util']:.0f}% util, {rec['gpu_mem_gib']:.1f} GiB",
                        flush=True,
                    )
                    t_log, audio_log, padded_log, loss_log, units_log = (
                        time.perf_counter(),
                        0.0,
                        0.0,
                        0.0,
                        0,
                    )
            model.eval()
            with torch.no_grad():
                val = evaluate(model)
            history.append(
                {"epoch": epoch + 1, "step": step, **{f"val_{k}": v for k, v in val.items()}}
            )
            improved = val["wer"] < best
            print(
                f"== epoch {epoch + 1}: val WER {val['wer']:.2f} ({sid(val)})  "
                f"CER {val['cer']:.2f}  raw WER {val['raw_wer']:.2f}  "
                f"loops {val['loops']}{retry_note(val)}" + ("  (best)" if improved else ""),
                flush=True,
            )
            (out / "history.json").write_text(json.dumps(history, indent=1))
            current = None
            if improved or resume is not None:  # a copy on the CPU: the run's only other one
                current = {
                    k: v.detach().to("cpu", copy=True) for k, v in model.state_dict().items()
                }
            if improved:
                best_state, best = current, val["wer"]
            if val["wer"] < counted - cfg.min_delta:  # at 0: the old rule, any strict gain
                counted, bad = val["wer"], 0
            else:
                bad += 1
            finished = bad >= cfg.patience
            if resume is not None:
                resume.save(
                    {
                        "epoch": epoch + 1,
                        "step": step,
                        "best": best,
                        "counted": counted,
                        "bad": bad,
                        "done": finished,
                        "history": history,
                        "model": current,
                        "best_model": None if improved else best_state,
                    }
                )
            del current
            if finished:
                print(
                    f"val WER gained less than {cfg.min_delta:g} points in {cfg.patience} "
                    "evaluations; stopping"
                )
                break
    except KeyboardInterrupt:
        # Colab's stop button: end training here and keep the best weights so far, so they are
        # saved and scored like a finished run's. Before any evaluation there are none to keep.
        if best_state is None:
            raise
        stopped_by_hand = True
        print(f"stopped by hand; keeping the best weights (val WER {best:.2f})", flush=True)
    monitor.window()
    with timed("saving the best weights"):
        if best_state is not None:
            model.load_state_dict(best_state)
        model.eval()
        save_best(model)
    (out / "config.json").write_text(json.dumps(asdict(cfg), indent=1))
    return {
        "best_val_wer": best,
        "steps": step,
        "history": history,
        "stopped_by_hand": stopped_by_hand,
    }

In [ ]:
%%writefile /content/ft/evalkit.py
"""One evaluation for every model (D105).

Whatever was trained -- Flex, a blend of its weights, a student at any stage -- is scored by the
functions here, so a number in one notebook means what it means in another:

  * gold and val, folded and raw, split into substitutions, deletions and insertions, overall and
    per clip class, and paired clip by clip against named reference runs with episodes resampled
    (`evaluate_run`);
  * the public Nepali sets, folded, raw and plain, paired against base Flex with each set's own
    unit resampled (`run_benchmarks`).

Every split and set also keeps its errors: each aligned word pair, classified, in
`harness/errors/<set>.parquet` with each clip's crosstalk and SNR, and a `breakdown` beside its
WER (crosstalk buckets, numbers, the
most common substitutions, deletions and insertions; docs/WER-Breakdown.md). Both come from the
harness's own `error_mining.py` and `error_store.py`, which the dataset's `harness/` copy carries
beside fold.py, so the Models page reads exactly what the notebook wrote.

The notebook hands in what is model-specific: `decode`, its batched decoder, and `score`,
`ftkit.harness_scorer`, which is the harness's own fold.py. No torch here, so the rules are tested
from backend/tests. Keep it importable on Python 3.10+.
"""

from __future__ import annotations

import datetime as dt
import hashlib
import io
import json
import shutil
import subprocess
import tarfile
import time
import unicodedata
from collections.abc import Callable, Collection, Iterator, Mapping, Sequence
from contextlib import contextmanager
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import distill
import sweep

SR = 16_000
#: Scored in this order: gold first, so a run stopped while scoring has the held-out number.
SPLITS = ("gold", "val")
BUCKETS = ("none", "0-5%", "5-15%", ">15%")
#: The clip classes every paired difference is also split by (D87).
REPORT_KEYS = ("overlap", "snr", "speakers", "cmi", "duration")
#: What a result row keeps of a split's metrics.
KEEP = ("wer", "raw_wer", "cer", "sub", "del", "ins", "loops", "clips", "rtf", "vs")

#: `decode(rows)` -> texts, per-clip compute seconds, and the loop retries as
#: (segment_id, first decode, retried decode).
Decode = Callable[[Sequence[dict]], tuple[list[str], list[float], list[tuple[str, str, str]]]]
Counts = Mapping[str, Sequence[int]]


@contextmanager
def timed(label: str) -> Iterator[None]:
    """`label...` before a step and `label: done in N s` after it: a stalled step names itself."""
    print(f"{label}...", flush=True)
    t0 = time.perf_counter()
    yield
    print(f"{label}: done in {time.perf_counter() - t0:.0f} s", flush=True)


def duration(row: dict) -> float:
    return row["end_time"] - row["start_time"]


# --- provenance ----------------------------------------------------------------------------------


def check_export(manifest: Mapping[str, Any], expected: str) -> None:
    """Refuse a dataset that is not the export every model in the comparison trains on.

    `expected` is the start of the training manifest's `exported_at` ("2026-09-30"). A date, not a
    commit: the dataset repo's history is squashed after each upload, which would orphan a pinned
    commit, and the export's own timestamp survives it."""
    got = str(manifest.get("exported_at") or "")
    if not expected or not got.startswith(expected):
        raise ValueError(
            f"the dataset is the {got or 'undated'} export, not {expected!r}: every model in one "
            "comparison trains and is scored on the same export"
        )


def kit_digests(folder: Path) -> dict[str, str]:
    """The first 16 hex digits of each kit file's sha256, for the model card: which code ran."""
    return {
        p.name: hashlib.sha256(p.read_bytes()).hexdigest()[:16]
        for p in sorted(Path(folder).glob("*.py"))
    }


# --- transcripts and per-clip counts -------------------------------------------------------------


def read_hyps(path: Path | str) -> dict[str, str]:
    """A transcript file (`write_hyps`) as {segment_id: text}."""
    with Path(path).open(encoding="utf-8") as fh:
        return {j["segment_id"]: j["text"] for j in map(json.loads, fh)}


def write_hyps(
    path: Path, rows: Sequence[dict], texts: Sequence[str], compute: Sequence[float]
) -> None:
    """One transcript per line, the format the harness's Models page imports (D83)."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as fh:
        for r, text, c in zip(rows, texts, compute, strict=True):
            row = {"segment_id": r["segment_id"], "text": text, "compute_s": c}
            fh.write(json.dumps(row, ensure_ascii=False) + "\n")


def counts(rows: Sequence[dict], clips: Sequence[Mapping[str, int]]) -> dict[str, list[int]]:
    """{segment_id: [folded errors, reference words]}: all a paired comparison needs of a run."""
    return {r["segment_id"]: [c["errors"], c["words"]] for r, c in zip(rows, clips, strict=True)}


def reference_counts(rows: Sequence[dict], by_id: Mapping[str, str], score: Any) -> dict:
    """Another system's transcripts scored against these rows' labels, on the clips it has.

    For a reference that left only transcripts (a run from an earlier export): its counts are
    taken against the current labels, and `pair` compares on the clips both scored."""
    have = [r for r in rows if r["segment_id"] in by_id]
    clips = score.per_clip([r["text"] for r in have], [by_id[r["segment_id"]] for r in have])
    return counts(have, clips)


def pair(
    rows: Sequence[dict],
    a: Counts,
    b: Counts,
    *,
    keys: Sequence[str] = REPORT_KEYS,
    group: str = "episode_id",
    n: int = 2000,
) -> dict[str, Any]:
    """WER(b) - WER(a) in points with a 95% interval, on the clips both runs scored.

    `group` names the unit resampled: a clip's episode on gold and val, a speaker, sentence or
    video on a public set. Returns `clips` (how many were shared), `all`, and one entry per value
    of each clip class in `keys`, as `[difference, low, high]`. Raises when no clip is shared."""
    idx = [i for i, r in enumerate(rows) if r["segment_id"] in a and r["segment_id"] in b]
    if not idx:
        raise ValueError("the two runs share no clip")

    def diff(which: Sequence[int]) -> list[float]:
        return list(
            sweep.paired_bootstrap(
                [
                    dict(zip(("errors", "words"), a[rows[i]["segment_id"]], strict=True))
                    for i in which
                ],
                [
                    dict(zip(("errors", "words"), b[rows[i]["segment_id"]], strict=True))
                    for i in which
                ],
                [str(rows[i][group]) for i in which],
                n=n,
            )
        )

    out: dict[str, Any] = {"clips": len(idx), "all": diff(idx)}
    for key in keys:
        values: dict[str, list[int]] = {}
        for i in idx:
            value = (rows[i].get("classes") or {}).get(key)
            if value is not None:
                values.setdefault(str(value), []).append(i)
        for value, which in sorted(values.items()):
            out[f"{key}={value}"] = diff(which)
    return out


# --- error mining (docs/WER-Breakdown.md) --------------------------------------------------------


def _miner() -> tuple[Any, Any] | None:
    """`error_mining` and `error_store` from the harness copy `ftkit.harness_scorer` put on the
    path, or None -- with a line saying so -- when the dataset's `harness/` predates them."""
    try:
        from app.services import error_mining, error_store
    except ImportError as exc:
        print(f"no error mining in the dataset's harness/ ({exc}): no error files are written")
        return None
    return error_mining, error_store


def write_errors(
    path: Path,
    run: str,
    rows: Sequence[dict],
    texts: Sequence[str],
    clips: Sequence[Any],
    *,
    overlap: Mapping[str, float | None] | None = None,
    snr: Mapping[str, float | None] | None = None,
    by: str | None = None,
) -> dict[str, Any] | None:
    """Every aligned word pair of one split or set, classified, written to `path` (its stem is
    the set's name); returns the file's report (blocks 1-3), or None when nothing was written.

    A clip's alignment is the one `score.per_clip` counted, when it kept it. `overlap` and `snr`
    are a public set's measured crosstalk and speech-to-noise ratio by clip id; gold and val rows
    carry their own `overlap_share` and `acoustics`."""
    kit = _miner()
    if kit is None:
        return None
    mining, store = kit
    ids = mining.unique_ids(str(r["segment_id"]) for r in rows)
    found = mining.rows(
        run,
        Path(path).stem,
        [
            {
                "clip_id": r["segment_id"],
                "group": r["episode_id"],
                "ref": r["text"],
                "hyp": text,
                "overlap_share": overlap.get(i) if overlap is not None else r.get("overlap_share"),
                "snr_db": snr.get(i)
                if snr is not None
                else (r.get("acoustics") or {}).get("snr_db"),
                "by": r.get(by) if by else None,
                "alignment": c.get("alignment") if isinstance(c, Mapping) else None,
            }
            for r, text, c, i in zip(rows, texts, clips, ids, strict=True)
        ],
    )
    if not found:
        return None
    mining.write(found, path)
    return store.report(Path(path))


#: A public set's measured recording conditions, each `benchmarks/<kind>/<set>.parquet`.
CONDITIONS = ("overlap", "acoustics")


def fetch_conditions(
    repo: str, token: str, folder: Path, names: Sequence[str] | None = None
) -> Path:
    """Each public set's measured crosstalk and acoustics, `benchmarks/{overlap,acoustics}/<set>
    .parquet` in `repo`, copied into `folder/overlap/` and `folder/acoustics/`
    (scripts/measure_benchmark_overlap.py wrote them); a set without one is unmeasured on it."""
    from huggingface_hub import hf_hub_download
    from huggingface_hub.utils import EntryNotFoundError

    folder = Path(folder)
    for kind in CONDITIONS:
        (folder / kind).mkdir(parents=True, exist_ok=True)
        for name in names or tuple(BENCHMARKS):
            try:
                got = hf_hub_download(repo, f"benchmarks/{kind}/{name}.parquet", token=token)
            except EntryNotFoundError:
                print(f"{name}: no measured {kind} in {repo}; its clips are unmeasured on it")
                continue
            shutil.copyfile(got, folder / kind / f"{name}.parquet")
    return folder


def print_breakdown(label: str, report: Mapping[str, Any] | None) -> None:
    """Block 1 and block 2 in two lines, under a WER line."""
    if not report:
        return
    buckets = " · ".join(
        f"{b['bucket']} {b['wer']:.2f} ({100 * b['share_of_errors']:.0f}% of errors)"
        for b in report["overlap"]
    )
    n = report["numbers"]
    print(f"    {label} crosstalk: {buckets}")
    print(
        f"    {label} numbers: {n['errors']} errors ({100 * n['share_of_errors']:.1f}%), "
        f"WER without them {n['wer_without']:.2f}"
    )


# --- gold and val --------------------------------------------------------------------------------


def score_split(
    rows: Sequence[dict],
    texts: Sequence[str],
    compute: Sequence[float],
    log: Sequence[tuple[str, str, str]],
    score: Any,
    *,
    errors: Path | None = None,
    run: str = "",
) -> tuple[dict[str, Any], dict[str, list[int]]]:
    """One split's metrics and its per-clip counts, each clip aligned once.

    Metrics: the folded, raw and character error rates with S/D/I, the real-time factor, the loop
    retries and the greedy-only score from the same decode, and every clip class (`by_class`).
    With `errors` (`harness/errors/<split>.parquet`), the split's error rows are written there and
    its `breakdown` added."""
    refs = [r["text"] for r in rows]
    clips = score.per_clip(refs, texts)
    m = score.summarize(clips)
    m["rtf"] = sum(compute) / max(sum(duration(r) for r in rows), 1e-9)
    m["retried"] = [{"segment_id": s, "first": f, "retry": t} for s, f, t in log]
    if log:  # the same run before any retry: only the retried clips are aligned again
        first = {s: f for s, f, _ in log}
        idx = [i for i, r in enumerate(rows) if r["segment_id"] in first]
        again = score.per_clip([refs[i] for i in idx], [first[rows[i]["segment_id"]] for i in idx])
        greedy = list(clips)
        for i, c in zip(idx, again, strict=True):
            greedy[i] = c
        m["greedy_only"] = score.summarize(greedy)
    m["by_class"] = distill.by_class(rows, clips, score.summarize)
    if errors is not None:
        m["breakdown"] = write_errors(errors, run, rows, texts, clips)
    return m, counts(rows, clips)


def evaluate_run(
    out: Path,
    run: str,
    *,
    splits: Mapping[str, Sequence[dict]],
    decode: Decode,
    score: Any,
    card: Mapping[str, Any],
    meta: Mapping[str, Any] | None = None,
    references: Mapping[str, Mapping[str, Counts]] | None = None,
    keys: Sequence[str] = REPORT_KEYS,
) -> dict[str, Any]:
    """Decode gold and val with the weights `decode` reads, score them, and write the run's files.

    Written under `out`: `harness/<split>.jsonl` and `harness/model_card.json` (what the Models
    page imports, D83), `harness/errors/<split>.parquet` (its error rows), `<split>_metrics.json`,
    `per_clip.json` (the counts another run is paired against) and `result.json`, whose presence
    on the hub marks the run as done. `references` is {name: {split: per-clip counts}}; each
    becomes `vs[name]`, this run minus that one. `meta` (recipe, seed, stage, best epoch, ...) goes
    into the card and the result row, which is returned."""
    out = Path(out)
    results: dict[str, dict] = {}
    per_clip: dict[str, dict] = {}
    for split in SPLITS:
        rows = splits[split]
        with timed(f"{run} {split}: decoding {len(rows)} clips"):
            texts, compute, log = decode(rows)
        with timed(f"{run} {split}: scoring, per clip class and against the references"):
            errors = out / "harness" / "errors" / f"{split}.parquet"
            m, per_clip[split] = score_split(
                rows, texts, compute, log, score, errors=errors, run=run
            )
            m["vs"] = {}
            for name, ref in (references or {}).items():
                try:
                    m["vs"][name] = pair(rows, ref.get(split) or {}, per_clip[split], keys=keys)
                except ValueError:  # the reference scored none of this split's clips
                    continue
        results[split] = m
        write_hyps(out / "harness" / f"{split}.jsonl", rows, texts, compute)
        (out / f"{split}_metrics.json").write_text(json.dumps(m, indent=1, ensure_ascii=False))
    gold, val = results["gold"], results["val"]
    full_card = {
        **card,
        **(meta or {}),
        "created_at": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),  # noqa: UP017
        "run_name": run,
        "val_wer": val["wer"],
        "gold_wer": gold["wer"],
        # folded, per 100 reference words; S + D + I = WER
        "val_sid": {k: val[k] for k in ("sub", "del", "ins")},
        "gold_sid": {k: gold[k] for k in ("sub", "del", "ins")},
    }
    (out / "harness" / "model_card.json").write_text(
        json.dumps(full_card, indent=1, ensure_ascii=False)
    )
    (out / "per_clip.json").write_text(json.dumps(per_clip))
    row = {
        "run_name": run,
        **(meta or {}),
        "val_wer": val["wer"],
        "gold_wer": gold["wer"],
        **{split: {k: results[split][k] for k in KEEP} for split in SPLITS},
        "gold_by_class": {k: gold["by_class"].get(k) for k in keys},
    }
    (out / "result.json").write_text(json.dumps(row, indent=1, ensure_ascii=False))
    for split in ("val", "gold"):
        m = results[split]
        print(
            f"{run} {split}: WER {m['wer']:.2f} (S {m['sub']:.2f}  D {m['del']:.2f}  "
            f"I {m['ins']:.2f}), raw {m['raw_wer']:.2f}"
        )
        for name, vs in m["vs"].items():
            d, lo, hi = vs["all"]
            print(f"    minus {name}: {d:+.2f} [{lo:+.2f}, {hi:+.2f}] on {vs['clips']} clips")
        print_breakdown(split, m.get("breakdown"))
    return row


# --- a finished run, scored again under newer rules --------------------------------------------

#: What `evaluate_run` derives from the scores, in `result.json` and the card: everything else in
#: them is the run's recipe, kept as it was when the run is scored again.
_ROW_SCORES = ("run_name", "val_wer", "gold_wer", "gold", "val", "gold_by_class")
_CARD_SCORES = (
    "created_at",
    "rescored_at",
    "run_name",
    "val_wer",
    "gold_wer",
    "val_sid",
    "gold_sid",
)


def stored_decode(out: Path) -> Decode:
    """A `decode` that reads a finished run's transcripts, compute and loop retries from its
    files instead of running a model. A clip the run never transcribed is refused: the rows are
    from another export."""
    out = Path(out)
    texts: dict[str, tuple[str, float]] = {}
    retries: dict[str, tuple[str, str, str]] = {}
    for split in SPLITS:
        hyps = out / "harness" / f"{split}.jsonl"
        if hyps.exists():
            with hyps.open(encoding="utf-8") as fh:
                for j in map(json.loads, fh):
                    texts[j["segment_id"]] = (j["text"], j["compute_s"])
        metrics = out / f"{split}_metrics.json"
        if metrics.exists():
            for r in json.loads(metrics.read_text("utf-8")).get("retried", []):
                retries[r["segment_id"]] = (r["segment_id"], r["first"], r["retry"])

    def decode(rows: Sequence[dict]) -> tuple[list[str], list[float], list[tuple[str, str, str]]]:
        missing = [r["segment_id"] for r in rows if r["segment_id"] not in texts]
        if missing:
            raise ValueError(
                f"{len(missing)} clip(s) this run never transcribed, first {missing[0]!r}: "
                "the rows are from another export"
            )
        ids = [r["segment_id"] for r in rows]
        return (
            [texts[i][0] for i in ids],
            [texts[i][1] for i in ids],
            [retries[i] for i in ids if i in retries],
        )

    return decode


def rescore_run(
    out: Path,
    run: str,
    *,
    splits: Mapping[str, Sequence[dict]],
    score: Any,
    fold_version: str | None = None,
    references: Mapping[str, Mapping[str, Counts]] | None = None,
    keys: Sequence[str] = REPORT_KEYS,
) -> dict[str, Any]:
    """Score a finished run again from its stored transcripts, under `score`'s rules (D113).

    The run's files (`evaluate_run`) are rewritten as if it had been scored today: the metrics,
    error rows, per-clip counts, result row and card, with the recipe kept from `result.json`, the
    card's `created_at` kept and `rescored_at` added, and `fold_version` (default:
    `score.fold_version`) set. Nothing is decoded."""
    out = Path(out)
    old_row = json.loads((out / "result.json").read_text("utf-8"))
    old_card = json.loads((out / "harness" / "model_card.json").read_text("utf-8"))
    meta = {k: v for k, v in old_row.items() if k not in _ROW_SCORES}
    card = {k: v for k, v in old_card.items() if k not in _CARD_SCORES and k not in meta}
    card["fold_version"] = fold_version or score.fold_version
    row = evaluate_run(
        out, run, splits=splits, decode=stored_decode(out), score=score, card=card, meta=meta,
        references=references, keys=keys,
    )  # fmt: skip
    path = out / "harness" / "model_card.json"
    full = json.loads(path.read_text("utf-8"))
    full["created_at"] = old_card.get("created_at", full["created_at"])
    full["rescored_at"] = dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds")  # noqa: UP017
    path.write_text(json.dumps(full, indent=1, ensure_ascii=False))
    return row


def rescore_benchmark(
    out: Path,
    name: str,
    *,
    score: Any,
    run: str | None = None,
    conditions_dir: Path | None = None,
) -> dict[str, Any]:
    """Score a public set again from its stored lines (`run_benchmarks`), under `score`'s rules.

    The lines hold each clip's reference, transcript, group and `by` value, which is all a score
    needs; the hours, speed, retries and limit of the decode are kept from the old summary.
    Rewrites `<name>.json`, `<name>.jsonl` and the set's error rows."""
    out = Path(out)
    folder = out / "benchmarks"
    old = json.loads((folder / f"{name}.json").read_text("utf-8"))
    with (folder / f"{name}.jsonl").open(encoding="utf-8") as fh:
        lines = [json.loads(line) for line in fh]
    by = BENCHMARKS[name].by
    rows = [
        {
            "segment_id": x["id"],
            "episode_id": x["group"],
            "text": x["ref"],
            "start_time": 0.0,
            "end_time": 0.0,
            **({by: x[by]} if by else {}),
        }
        for x in lines
    ]
    summary, new_lines = score_benchmark(
        name,
        rows,
        [x["hyp"] for x in lines],
        score,
        errors=out / "harness" / "errors" / f"{name}.parquet",
        run=run or out.name,
        overlap=_measured(conditions_dir, "overlap", name),
        snr=_measured(conditions_dir, "acoustics", name),
    )
    for key in ("hours", "retried", "x_realtime", "limit"):
        if key in old:
            summary[key] = old[key]
    with (folder / f"{name}.jsonl").open("w", encoding="utf-8") as fh:
        fh.writelines(json.dumps(line, ensure_ascii=False) + "\n" for line in new_lines)
    (folder / f"{name}.json").write_text(json.dumps(summary, indent=1, ensure_ascii=False))
    print(f"{name}: WER {old['wer']:.2f} -> {summary['wer']:.2f}")
    return summary


# --- the public sets -----------------------------------------------------------------------------


@dataclass(frozen=True)
class Benchmark:
    """One public Nepali test set, as it was scored on 2026-09-27 (findings.md).

    `audio`, `text` and `group` are its columns: the clip, the reference, and the unit a paired
    interval resamples (a speaker; FLEURS repeats sentences, so there it is the sentence; nepali_cs
    is resampled by video). `clip_id` names each clip, from one column or several joined by `-`.
    `by` is a column the score is also split by. `base_wer` is base Flex's folded WER on
    2026-09-27: decoding base again should land on it, which is the loader's check."""

    name: str
    what: str
    repo: str
    config: str
    split: str
    audio: str
    text: str
    group: str
    clip_id: tuple[str, ...]
    base_wer: float
    by: str | None = None
    #: "parquet": the hub's converted files. "tar": Common Voice's mirror, a TSV and a tar of MP3s.
    layout: str = "parquet"


BENCHMARKS: dict[str, Benchmark] = {
    b.name: b
    for b in (
        Benchmark(
            name="fleurs",
            what="FLEURS ne_np test: read Wikipedia sentences",
            repo="google/fleurs",
            config="ne_np",
            split="test",
            audio="audio",
            text="raw_transcription",
            group="id",
            clip_id=("path",),
            base_wer=11.10,
        ),
        Benchmark(
            name="slr54",
            what="OpenSLR 54 test: crowd-sourced read speech (the mirror's own split)",
            repo="iamTangsang/OpenSLR54-Nepali-ASR",
            config="default",
            split="test",
            audio="utterance",
            text="transcription",
            group="speaker_id",
            clip_id=("utterance_id",),
            base_wer=8.17,
        ),
        Benchmark(
            name="common_voice",
            what="Common Voice 22 ne-NP test: crowd-sourced read speech",
            repo="fsicoli/common_voice_22_0",
            config="ne-NP",
            split="test",
            audio="path",
            text="sentence",
            group="client_id",
            clip_id=("path",),
            base_wer=8.78,
            layout="tar",
        ),
        Benchmark(
            name="indicvoices",
            what="IndicVoices nepali valid: read, extempore and conversation",
            repo="ai4bharat/IndicVoices",
            config="nepali",
            split="valid",
            audio="audio_filepath",
            text="text",
            group="speaker_id",
            clip_id=(),  # it has none: clips are numbered in file order
            base_wer=12.70,
            by="scenario",
        ),
        Benchmark(
            name="nepali_cs",
            what="nepali-cs-asr test: Nepali-English code-switched lectures",
            repo="saileshbro/nepali-cs-asr",
            config="default",
            split="test",
            audio="audio",
            text="transcription",
            group="video_id",
            clip_id=("video_id", "start_ms"),
            base_wer=10.98,
            by="speech_type",
        ),
    )
}


def plain_tokens(text: str | None) -> list[str]:
    """The normalisation published Nepali results use: NFC, punctuation and symbols dropped,
    Latin lowercased, and nothing else. No fold: `टिम` and `team` are different words here."""
    text = unicodedata.normalize("NFC", text or "")
    kept = "".join(c if unicodedata.category(c)[0] in "LMN" else " " for c in text)
    return kept.lower().split()


def edit_distance(ref: Sequence[str], hyp: Sequence[str]) -> int:
    """Word-level Levenshtein distance: the errors of a plain WER."""
    row = list(range(len(hyp) + 1))
    for i, r in enumerate(ref, 1):
        prev, row[0] = row[0], i
        for j, h in enumerate(hyp, 1):
            prev, row[j] = row[j], min(row[j] + 1, row[j - 1] + 1, prev + (r != h))
    return row[-1]


def plain_wer(refs: Sequence[str], hyps: Sequence[str]) -> float:
    """Pooled plain WER in percent: errors over reference words, across all clips."""
    errors = words = 0
    for ref, hyp in zip(refs, hyps, strict=True):
        r = plain_tokens(ref)
        errors += edit_distance(r, plain_tokens(hyp))
        words += len(r)
    return 100 * errors / max(words, 1)


def _audio(data: bytes) -> Any:
    """Any encoded clip as 16 kHz mono int16: libsndfile first, ffmpeg for what it cannot read."""
    import numpy as np
    import soundfile as sf

    try:
        x, sr = sf.read(io.BytesIO(data), dtype="float32", always_2d=True)
        x = x.mean(axis=1)
        if sr != SR:
            from math import gcd

            from scipy.signal import resample_poly

            g = gcd(SR, sr)
            x = resample_poly(x, SR // g, sr // g)
        return np.round(np.clip(x, -1.0, 1.0) * 32767).astype(np.int16)
    except Exception:
        cmd = [
            "ffmpeg",
            "-v",
            "error",
            "-i",
            "pipe:0",
            "-f",
            "s16le",
            "-ac",
            "1",
            "-ar",
            str(SR),
            "pipe:1",
        ]
        raw = subprocess.run(cmd, input=data, capture_output=True, check=True).stdout
        return np.frombuffer(raw, dtype=np.int16).copy()


def benchmark_row(bench: Benchmark, record: Mapping[str, Any], index: int, audio: Any) -> dict:
    """One clip as the decoders and the scorer read a row: it carries its own audio, its group
    stands where an episode does, and `segment_id` is the id the 2026-09-27 files used."""
    if bench.clip_id:
        clip_id = "-".join(str(record[c]) for c in bench.clip_id).rsplit("/", 1)[-1]
    else:
        clip_id = f"iv-{index:05d}"
    row = {
        "segment_id": clip_id,
        "episode_id": str(record[bench.group]),
        "text": record[bench.text] or "",
        "audio": audio,
        "start_time": 0.0,
        "end_time": len(audio) / SR,
    }
    if bench.by:
        row[bench.by] = record[bench.by]
    return row


def _parquet_records(bench: Benchmark, work: Path, token: str) -> Iterator[tuple[dict, bytes]]:
    """(record, encoded audio) from the hub's parquet conversion of the split, a file at a time."""
    import pyarrow.parquet as pq
    import requests

    headers = {"Authorization": f"Bearer {token}"}
    listing = (
        f"https://huggingface.co/api/datasets/{bench.repo}/parquet/{bench.config}/{bench.split}"
    )
    urls = requests.get(listing, headers=headers, timeout=60)
    urls.raise_for_status()
    columns = sorted(
        {bench.audio, bench.text, bench.group, *bench.clip_id, *([bench.by] if bench.by else [])}
    )
    for k, url in enumerate(urls.json()):
        path = work / f"{bench.name}-{k:04d}.parquet"
        if not path.exists():
            with requests.get(url, headers=headers, stream=True, timeout=600) as r:
                r.raise_for_status()
                with path.with_suffix(".part").open("wb") as fh:
                    for chunk in r.iter_content(1 << 22):
                        fh.write(chunk)
            path.with_suffix(".part").rename(path)
        table = pq.ParquetFile(path)
        missing = set(columns) - set(table.schema_arrow.names)
        if missing:
            raise KeyError(
                f"{bench.repo}: no column {sorted(missing)}; it has {table.schema_arrow.names}"
            )
        for batch in table.iter_batches(batch_size=64, columns=columns):
            for record in batch.to_pylist():
                yield record, record[bench.audio]["bytes"]


def _tar_records(bench: Benchmark, token: str) -> Iterator[tuple[dict, bytes]]:
    """(record, encoded audio) from the Common Voice mirror: `transcript/<locale>/<split>.tsv`
    and the MP3s in `audio/<locale>/<split>/<locale>_<split>_<n>.tar`, in the TSV's order."""
    import csv

    from huggingface_hub import hf_hub_download, list_repo_files

    def fetch(name: str) -> str:
        return hf_hub_download(bench.repo, name, repo_type="dataset", token=token)

    tsv = fetch(f"transcript/{bench.config}/{bench.split}.tsv")
    with open(tsv, encoding="utf-8", newline="") as fh:
        records = list(csv.DictReader(fh, delimiter="\t", quoting=csv.QUOTE_NONE))
    prefix = f"audio/{bench.config}/{bench.split}/"
    clips: dict[str, bytes] = {}
    for name in sorted(list_repo_files(bench.repo, repo_type="dataset", token=token)):
        if name.startswith(prefix) and name.endswith(".tar"):
            with tarfile.open(fetch(name)) as tf:
                for member in tf:
                    if member.isfile():
                        clips[member.name.rsplit("/", 1)[-1]] = tf.extractfile(member).read()
    for record in records:
        yield record, clips[record[bench.audio]]


def iter_benchmark(name: str, work: Path, token: str) -> Iterator[dict]:
    """A public set's clips one at a time, in file order, each a row with its audio (16 kHz mono
    int16): for a pass that must not hold the whole set in RAM."""
    bench = BENCHMARKS[name]
    work = Path(work)
    work.mkdir(parents=True, exist_ok=True)
    records = (
        _tar_records(bench, token)
        if bench.layout == "tar"
        else _parquet_records(bench, work, token)
    )
    for index, (record, data) in enumerate(records):
        yield benchmark_row(bench, record, index, _audio(data))


def load_benchmark(name: str, work: Path, token: str, limit: int | None = None) -> list[dict]:
    """A public set's clips as rows with their audio in RAM (16 kHz mono int16), in file order.

    `limit` keeps the first clips only, for a smoke run. Nothing here is cached beyond the
    downloaded files: a set is loaded, decoded, scored and dropped."""
    rows = []
    for row in iter_benchmark(name, work, token):
        if limit is not None and len(rows) >= limit:
            break
        rows.append(row)
    return rows


def score_benchmark(
    name: str,
    rows: Sequence[dict],
    texts: Sequence[str],
    score: Any,
    *,
    errors: Path | None = None,
    run: str = "",
    overlap: Mapping[str, float | None] | None = None,
    snr: Mapping[str, float | None] | None = None,
) -> tuple[dict[str, Any], list[dict]]:
    """A public set's summary and its per-clip lines.

    Summary: folded WER with S/D/I, raw WER and CER from the harness's scorer, the plain WER, and
    the score per value of the set's `by` column. A line is `{id, group, ref, hyp, errors,
    words}`: the 2026-09-27 files' fields plus the counts a paired comparison needs. With
    `errors`, the set's error rows are written there, crosstalk from `overlap` and SNR from `snr`
    (clips missing from either are unmeasured on it), and its `breakdown` added to the summary."""
    bench = BENCHMARKS[name]
    refs = [r["text"] for r in rows]
    clips = score.per_clip(refs, texts)
    summary = score.summarize(clips)
    summary["plain_wer"] = plain_wer(refs, texts)
    summary["hours"] = sum(duration(r) for r in rows) / 3600
    if bench.by:
        values: dict[str, list[int]] = {}
        for i, r in enumerate(rows):
            values.setdefault(str(r[bench.by]), []).append(i)
        summary["by"] = {
            v: score.summarize([clips[i] for i in idx]) for v, idx in sorted(values.items())
        }
    if errors is not None:
        summary["breakdown"] = write_errors(
            errors, run, rows, texts, clips, overlap=overlap or {}, snr=snr or {}, by=bench.by
        )
    lines = []
    for r, hyp, c in zip(rows, texts, clips, strict=True):
        line = {
            "id": r["segment_id"],
            "group": r["episode_id"],
            "ref": r["text"],
            "hyp": hyp,
            "errors": c["errors"],
            "words": c["words"],
        }
        if bench.by:
            line[bench.by] = r[bench.by]
        lines.append(line)
    return summary, lines


def pair_benchmark(a: Sequence[dict], b: Sequence[dict], n: int = 2000) -> dict[str, Any]:
    """WER(b) - WER(a) on one public set from two models' per-clip lines, resampling the set's
    group; `{clips, all: [difference, low, high]}`."""
    rows = [{"segment_id": x["id"], "episode_id": x["group"]} for x in b]
    return pair(
        rows,
        {x["id"]: [x["errors"], x["words"]] for x in a},
        {x["id"]: [x["errors"], x["words"]] for x in b},
        keys=(),
        n=n,
    )


def _measured(folder: Path | None, kind: str, name: str) -> dict[str, float | None] | None:
    """`folder/<kind>/<name>.parquet` as values by clip id, or None when absent -- or when the
    harness copy predates the reader (SNR came with mine-v2)."""
    path = Path(folder) / kind / f"{name}.parquet" if folder is not None else None
    if path is None or not path.exists() or (kit := _miner()) is None:
        return None
    read = getattr(kit[0], "read_overlap" if kind == "overlap" else "read_snr", None)
    return read(path) if read is not None else None


def run_benchmarks(
    out: Path,
    *,
    decode: Decode,
    score: Any,
    work: Path,
    token: str,
    names: Sequence[str] = tuple(BENCHMARKS),
    limit: int | None = None,
    done: Collection[str] = (),
    run: str | None = None,
    conditions_dir: Path | None = None,
) -> dict[str, dict]:
    """Decode and score each public set with the weights `decode` reads, one set at a time.

    Writes `out/benchmarks/<name>.jsonl` (per clip) and `<name>.json` (summary), and the set's
    error rows to `out/harness/errors/<name>.parquet`, as each set finishes, and skips a set named
    in `done`, so a lost runtime costs one set. `run` names the rows (default: `out`'s folder);
    `conditions_dir` holds the sets' measured crosstalk and acoustics (`fetch_conditions`)."""
    run = run or Path(out).name
    folder = Path(out) / "benchmarks"
    folder.mkdir(parents=True, exist_ok=True)
    summaries = {}
    for name in names:
        if name in done:
            print(f"{name}: already scored, skipped")
            continue
        with timed(f"{name}: loading"):
            rows = load_benchmark(name, work, token, limit)
        with timed(f"{name}: decoding {len(rows)} clips"):
            texts, compute, log = decode(rows)
        with timed(f"{name}: scoring"):
            summary, lines = score_benchmark(
                name,
                rows,
                texts,
                score,
                errors=Path(out) / "harness" / "errors" / f"{name}.parquet",
                run=run,
                overlap=_measured(conditions_dir, "overlap", name),
                snr=_measured(conditions_dir, "acoustics", name),
            )
        summary["retried"] = len(log)
        summary["x_realtime"] = sum(duration(r) for r in rows) / max(sum(compute), 1e-9)
        summary["limit"] = limit
        with (folder / f"{name}.jsonl").open("w", encoding="utf-8") as fh:
            fh.writelines(json.dumps(line, ensure_ascii=False) + "\n" for line in lines)
        (folder / f"{name}.json").write_text(json.dumps(summary, indent=1, ensure_ascii=False))
        print(
            f"{name}: WER {summary['wer']:.2f} (S {summary['sub']:.2f}  D {summary['del']:.2f}  "
            f"I {summary['ins']:.2f}), plain {summary['plain_wer']:.2f}, {len(rows)} clips, "
            f"{summary['x_realtime']:.0f}x realtime"
        )
        print_breakdown(name, summary.get("breakdown"))
        summaries[name] = summary
        del rows
    return summaries

In [ ]:
%%writefile /content/ft/sweep.py
"""Which run's weights are kept, and how two runs are compared (D96, D105).

A notebook trains several runs on the same export: the crosstalk sweep's `(XTALK_P, seed)` points
(D96), the augmentation ablation's named recipes, or the weight blends of one fine-tune (D105).
Each winner is chosen by val WER alone, under a rule fixed before any result was read. For
training runs: a recipe has to beat the baseline by more than the seed noise, measured as the gap
between the baseline's two seeds, or the baseline is kept. For named recipes that gap is never
taken as less than `MIN_NOISE` (D109). Gold never takes part in a choice, so every gold number
stays a held-out score.

Runs are compared on gold with a paired bootstrap that resamples whole episodes: clips of one
episode share a room and voices, and resampling them one by one would pretend to more independent
evidence than gold holds (findings.md, *How big gold has to be*). Pure numpy; importable without
torch.
"""

from __future__ import annotations

from collections.abc import Sequence

import numpy as np


def run_name(prefix: str, p: float, seed: int) -> str:
    """`<prefix>-p30-s0` for XTALK_P = 0.3 and seed 0: the run's folder and Models page id."""
    return f"{prefix}-p{round(p * 100):02d}-s{seed}"


def _winner(
    base: list[dict], other: list[dict], base_name: str, label, floor: float = 0.0
) -> tuple[dict, str]:
    """The rule every chooser shares. `base` and `other` are sorted by val WER, best first.

    The bar is the gap between the baseline's two best seeds, or `floor` when that is larger
    (D109). Both are rounded to 1e-6 so a margin equal to the bar is a tie, not a float win."""
    if not base or not other:
        best = (base or other)[0]
        return best, "only one kind of run, so the lowest val WER"
    if len(base) >= 2:
        noise = base[1]["val_wer"] - base[0]["val_wer"]
        seeds = f"{base_name} seeds {base[0]['seed']} and {base[1]['seed']}"
        noise_note = f"seed noise {noise:.2f} ({seeds})"
    else:
        noise = 0.0
        noise_note = f"seed noise unmeasured (one {base_name} seed)"
        if not floor:
            noise_note += ", so the rule is strict"
    if floor > noise:
        noise, noise_note = floor, f"the floor {floor:.2f}, as {noise_note} is below it"
    margin = base[0]["val_wer"] - other[0]["val_wer"]
    if round(margin, 6) > round(noise, 6):
        return other[
            0
        ], f"{label(other[0])} beats {base_name} on val by {margin:.2f} > {noise_note}"
    return base[0], f"best augmented margin {margin:.2f} is within {noise_note}; {base_name} kept"


def choose_winner(rows: Sequence[dict]) -> tuple[dict, str]:
    """The run whose weights are kept, and why, from rows with `xtalk_p`, `seed` and `val_wer`.

    p = 0 is its better seed. The noise is the gap between its two best seeds; with one p = 0 seed
    it is unmeasured and the rule is strict. The best augmented run wins only if it beats p = 0 by
    more than the noise; a tie goes to p = 0, the simpler recipe."""
    if not rows:
        raise ValueError("no runs to choose from")
    base = sorted((r for r in rows if r["xtalk_p"] == 0), key=lambda r: r["val_wer"])
    aug = sorted((r for r in rows if r["xtalk_p"] != 0), key=lambda r: r["val_wer"])
    return _winner(base, aug, "p=0", lambda r: f"p={r['xtalk_p']}")


#: The recipe every augmentation is measured against (D105): today's training, nothing added.
BASELINE = "vanilla"

#: The least a named recipe has to beat the baseline by on val, in WER points (D109). Two seeds
#: can agree by luck: 03a's differed by 0.02 on val and 0.08 on gold. Below this the seed gap is
#: not trusted as the noise.
MIN_NOISE = 0.15


def choose_recipe(
    rows: Sequence[dict], baseline: str = BASELINE, floor: float = MIN_NOISE
) -> tuple[dict, str]:
    """`choose_winner` for named recipes (D105): rows carry `recipe`, `seed` and `val_wer`.

    The baseline is its better seed, and the noise is the gap between its two best seeds, or
    `floor` when that is larger (D109). Another recipe wins only if it beats the baseline on val
    by more than that; otherwise the baseline is kept. Gold and the public sets never take part."""
    if not rows:
        raise ValueError("no runs to choose from")
    base = sorted((r for r in rows if r["recipe"] == baseline), key=lambda r: r["val_wer"])
    other = sorted((r for r in rows if r["recipe"] != baseline), key=lambda r: r["val_wer"])
    return _winner(base, other, baseline, lambda r: r["recipe"], floor)


def beats_baseline(
    row: dict, baseline_rows: Sequence[dict], baseline: str = BASELINE, floor: float = MIN_NOISE
) -> tuple[bool, str]:
    """Whether one run clears the rule on its own, and why: what decides, as each ablation run
    finishes, if its stage goes into the combined recipe and its weights are kept."""
    base = [r for r in baseline_rows if r["recipe"] == baseline]
    if not base:
        raise ValueError(f"no {baseline} run to measure against")
    winner, why = choose_recipe([*base, row], baseline, floor)
    return winner is row, why


def choose_blend(rows: Sequence[dict], tolerance: float = 0.3) -> tuple[dict, str]:
    """The weight blend that is kept, and why, from rows with `alpha` and `val_wer`.

    A blend is `(1 - alpha) * base + alpha * fine-tuned` (WiSE-FT), so alpha = 1 is the fine-tuned
    model and must be among the rows. The rule, fixed on 2026-09-27 before any blend was scored:
    the blend closest to base whose val WER is within `tolerance` points of the fine-tuned
    model's. Base itself (alpha = 0) is never a candidate, and if no blend qualifies the
    fine-tuned model is kept. Gold and the public sets never take part."""
    tuned = [r for r in rows if r["alpha"] == 1]
    if len(tuned) != 1:
        raise ValueError("the rule needs exactly one alpha = 1 row, the fine-tuned model")
    limit = tuned[0]["val_wer"] + tolerance
    ok = sorted(
        (r for r in rows if 0 < r["alpha"] < 1 and r["val_wer"] <= limit), key=lambda r: r["alpha"]
    )
    if not ok:
        return tuned[0], f"no blend is within {tolerance:g} of the fine-tuned val WER; it is kept"
    return ok[0], (
        f"alpha={ok[0]['alpha']:g} is the blend closest to base with val WER "
        f"{ok[0]['val_wer']:.2f} <= {tuned[0]['val_wer']:.2f} + {tolerance:g}"
    )


def paired_bootstrap(
    a: Sequence[dict],
    b: Sequence[dict],
    episodes: Sequence[str],
    *,
    n: int = 2000,
    seed: int = 0,
) -> tuple[float, float, float]:
    """WER(b) - WER(a) in points, with a 95% interval from resampling episodes.

    `a` and `b` are per-clip `{"errors", "words"}` for the same clips in the same order, and
    `episodes` names each clip's episode. WER is pooled (sum of errors over sum of words)."""
    if not (len(a) == len(b) == len(episodes)):
        raise ValueError("a, b and episodes must describe the same clips")
    ids = sorted(set(episodes))
    index = {e: i for i, e in enumerate(ids)}
    ep = np.array([index[e] for e in episodes])
    per = np.zeros((len(ids), 3))  # errors of a, errors of b, reference words
    clips = [[x["errors"], y["errors"], x["words"]] for x, y in zip(a, b, strict=True)]
    np.add.at(per, ep, np.array(clips))

    def diff(t: np.ndarray) -> float:
        return float(100 * (t[1] - t[0]) / max(t[2], 1))

    point = diff(per.sum(axis=0))
    rng = np.random.default_rng(seed)
    draws = rng.integers(0, len(ids), size=(n, len(ids)))
    totals = per[draws].sum(axis=1)  # (n, 3)
    boot = 100 * (totals[:, 1] - totals[:, 0]) / np.maximum(totals[:, 2], 1)
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return point, float(lo), float(hi)

In [ ]:
%%writefile /content/ft/distill.py
"""Pure helpers for the distillation notebook (roadmap §B).

No torch and no NeMo, so the rules that decide what shapes the students' tokenizer and what their
scores are paired against are tested from backend/tests. Keep it importable on Python 3.10+.
"""

from __future__ import annotations

import re
import unicodedata
from collections.abc import Callable, Mapping, Sequence
from typing import Any


def tokenizer_texts(splits: Mapping[str, Sequence[dict]]) -> list[str]:
    """The labels the shared student tokenizer is trained on: train only.

    Val and gold text never shapes the vocabulary, so a word only they contain is scored as the
    student would meet it on new audio. Raises if a val or gold clip is in train."""
    held = {r["segment_id"] for name in ("val", "gold") for r in splits.get(name, ())}
    leaked = sorted(held & {r["segment_id"] for r in splits["train"]})
    if leaked:
        raise ValueError(f"held-out clips in train: {leaked[:5]}")
    return [r["text"] for r in splits["train"]]


def reference_texts(rows: Sequence[dict], by_id: Mapping[str, str]) -> list[str]:
    """Another system's transcripts in `rows`' order, for pairing clip by clip. Raises, naming
    them, if any row has none: a pairing over a subset would not be the same clips."""
    missing = [r["segment_id"] for r in rows if r["segment_id"] not in by_id]
    if missing:
        raise ValueError(f"{len(missing)} clip(s) have no reference transcript: {missing[:5]}")
    return [by_id[r["segment_id"]] for r in rows]


def by_class(
    rows: Sequence[dict],
    clips: Sequence[Any],
    summarize: Callable[[list[Any]], Any],
    keys: Sequence[str] | None = None,
) -> dict[str, dict[str, Any]]:
    """`summarize` per value of each clip class (D87), as {key: {value: summary}}.

    `clips` holds one per-clip count per row (`score.per_clip`), aligned once by the caller: a
    group's score is its clips' counts added up, so no clip is aligned again per key. `keys`
    defaults to every class key any row carries. A clip without a key (or with no classes) is left
    out of that key's groups, not counted under a made-up value."""
    if len(rows) != len(clips):
        raise ValueError("rows and clips must describe the same clips")
    if keys is None:
        keys = sorted({k for r in rows for k in (r.get("classes") or {})})
    out: dict[str, dict[str, Any]] = {}
    for key in keys:
        groups: dict[str, list[int]] = {}
        for i, r in enumerate(rows):
            value = (r.get("classes") or {}).get(key)
            if value is not None:
                groups.setdefault(str(value), []).append(i)
        if groups:
            out[key] = {v: summarize([clips[i] for i in idx]) for v, idx in sorted(groups.items())}
    return out


def filter_pseudo(
    rows: Sequence[dict],
    *,
    tokens_per_s: tuple[float, float],
    drop_fraction: float,
    max_overlap_share: float | None = None,
) -> tuple[list[dict], dict[str, Any]]:
    """Roadmap §B step 3: keep the teacher's labels worth training on, cheapest rule first.

    Each row carries ``segment_id``, ``text``, ``n_tokens`` (teacher tokens), ``duration`` (s),
    ``mean_logprob`` and ``looped`` (the greedy output loops, ``ftkit.is_loop``), and, once
    PreDistill has measured it, ``overlap_share``. Dropped, in order: a clip overlapped for more
    than ``max_overlap_share`` of its length (crosstalk is where the teacher is weakest; ``None``
    applies no such rule, and a clip never measured is kept and counted); a clip whose output
    loops; an empty transcript; a rate outside ``tokens_per_s`` (the range train's labels span, in
    the teacher's tokens); then the least confident ``drop_fraction`` of what is left, by mean
    log-prob. Returns the kept rows, in input order, and a report of what each rule dropped.
    """
    if not 0.0 <= drop_fraction < 1.0:
        raise ValueError(f"drop_fraction must be in [0, 1), not {drop_fraction}")
    if max_overlap_share is not None and not 0.0 <= max_overlap_share <= 1.0:
        raise ValueError(f"max_overlap_share must be in [0, 1], not {max_overlap_share}")
    lo, hi = tokens_per_s
    dropped = {"overlap": 0, "loop": 0, "empty": 0, "rate": 0, "confidence": 0}
    unmeasured = sum(r.get("overlap_share") is None for r in rows)
    left = []
    for r in rows:
        share = r.get("overlap_share")
        if max_overlap_share is not None and share is not None and share > max_overlap_share:
            dropped["overlap"] += 1
        elif r["looped"]:
            dropped["loop"] += 1
        elif not r["text"].strip():
            dropped["empty"] += 1
        elif not lo <= r["n_tokens"] / r["duration"] <= hi:
            dropped["rate"] += 1
        else:
            left.append(r)
    cut = None
    n_drop = int(len(left) * drop_fraction)
    if n_drop:
        least = sorted(left, key=lambda r: r["mean_logprob"])[:n_drop]
        cut = max(r["mean_logprob"] for r in least)
        gone = {r["segment_id"] for r in least}
        left = [r for r in left if r["segment_id"] not in gone]
        dropped["confidence"] = n_drop
    report = {
        "total": len(rows),
        "kept": len(left),
        "kept_hours": round(sum(r["duration"] for r in left) / 3600, 3),
        "dropped": dropped,
        "logprob_cut": cut,
        "tokens_per_s": [lo, hi],
        "drop_fraction": drop_fraction,
        "max_overlap_share": max_overlap_share,
        "overlap_unmeasured": unmeasured,
    }
    return left, report


def latin_share(text: str) -> float:
    """The share of a transcript's words written in Latin script: a code-mixing proxy that needs
    only the text, for pseudo-labels, which have no CMI class."""
    words = [w for w in text.split() if any(c.isalpha() for c in w)]
    if not words:
        return 0.0
    return sum(any("a" <= c.lower() <= "z" for c in w) for w in words) / len(words)


def mixing_bucket(share: float) -> str:
    """``latin_share`` in the corpus's CMI buckets (D87): 0, <15, 15-30, 30+ percent."""
    if share == 0:
        return "0"
    if share < 0.15:
        return "<15"
    return "15-30" if share < 0.30 else "30+"


# --- PreDistill: the owner's zip of recordings (D101) ------------------------------------------

_NUMBER = re.compile(r"_\d+$")


def _path_safe(text: str) -> str:
    """Letters and digits of any script, combining marks (Devanagari's vowel signs, which regex's
    word class misses), ``.``, ``-`` and ``_`` kept; every other run of characters becomes one
    ``_``."""
    kept = "".join(
        c if c.isalnum() or c in "._-" or unicodedata.category(c).startswith("M") else "\0"
        for c in text
    )
    return re.sub("\0+", "_", kept).strip("_")


def source_from_filename(name: str) -> tuple[str, str]:
    """A recording named ``<channel_name>_<NN>.mp3`` as ``(source_id, channel)``.

    The source id is the file's stem with every run of characters unsafe in a path turned into
    one ``_`` (letters of any script, digits, ``.``, ``-`` and ``_`` are kept); the channel is the
    id less its trailing ``_<number>``.
    """
    stem = name.replace("\\", "/").rsplit("/", 1)[-1].rsplit(".", 1)[0]
    source_id = _path_safe(stem)
    return source_id, _NUMBER.sub("", source_id) or source_id


def channel_blocked(channel: str, blocked: Sequence[str]) -> bool:
    """Whether a channel's name contains a blocked name, ignoring case and ``_`` against spaces."""
    name = channel.replace("_", " ").casefold()
    return any(b.strip().casefold() in name for b in blocked if b.strip())


def slice_rows(source_id: str, channel: str, slices: Sequence[tuple[float, float]]) -> list[dict]:
    """The manifest rows of one recording's VAD slices, named and timed as the labelled export's
    rows are, so ``ftkit.AudioStore`` cuts them from the whole recording."""
    return [
        {
            "segment_id": f"{source_id}_{i:05d}",
            "episode_id": source_id,
            "source_id": source_id,
            "channel": channel,
            "start_time": round(start, 3),
            "end_time": round(end, 3),
            "duration": round(end - start, 3),
        }
        for i, (start, end) in enumerate(slices)
    ]


# --- overlapped speech on the unlabelled corpus (D105) -------------------------------------------
#
# The harness measures overlap per episode (app/services/overlap.py) and classes a clip by the
# share of it that is overlapped (app/services/clip_classes.py). The second module imports
# SQLAlchemy and the models, so it cannot be fetched into Colab; its two pure functions are
# repeated here, and a test keeps them equal to the harness's.


def spans_within(
    spans: Sequence[Sequence[float]], start: float, end: float
) -> list[tuple[float, float]]:
    """A recording's overlap spans cut to one clip, clip-relative, as the labelled export's
    ``overlap_spans`` are. Empty when the clip holds none."""
    out: list[tuple[float, float]] = []
    for lo, hi in spans:
        a, b = max(lo, start), min(hi, end)
        if b > a:
            out.append((round(a - start, 3), round(b - start, 3)))
    return out


def overlap_share(spans: Sequence[Sequence[float]] | None, duration: float) -> float | None:
    """The fraction of a clip spent in crosstalk; ``None`` when it was never measured."""
    if spans is None:
        return None
    if duration <= 0:
        return 0.0
    return min(1.0, sum(max(0.0, end - start) for start, end in spans) / duration)


def overlap_bucket(share: float | None) -> str:
    """The overlap bucket of a clip's overlap share, the corpus's clip class (D87)."""
    if share is None:
        return "unmeasured"
    if share <= 0:
        return "none"
    if share < 0.05:
        return "0-5%"
    if share <= 0.15:
        return "5-15%"
    return ">15%"


def with_overlap(rows: Sequence[dict], spans: Sequence[Sequence[float]] | None) -> list[dict]:
    """One recording's clip rows with ``overlap_spans`` (clip-relative) and ``overlap_share``,
    from the recording's overlap spans. ``None`` spans (no detector) leave both ``None``: never
    measured, which is not the same as clean."""
    out = []
    for r in rows:
        mine = None if spans is None else spans_within(spans, r["start_time"], r["end_time"])
        share = overlap_share(mine, r["end_time"] - r["start_time"])
        out.append(
            {
                **r,
                "overlap_spans": None if mine is None else [list(s) for s in mine],
                "overlap_share": None if share is None else round(share, 4),
            }
        )
    return out


def overlap_loss(rows: Sequence[dict], thresholds: Sequence[float]) -> dict[str, dict[str, Any]]:
    """What a ``max_overlap_share`` would cost, per channel and for ``all``: the hours of speech,
    the hours never measured, and the hours dropped at each threshold (a clip is dropped when its
    share is *above* the threshold, as ``filter_pseudo`` drops it). The table the owner reads
    before choosing the threshold."""
    out: dict[str, dict[str, Any]] = {}
    for r in rows:
        hours = (r["end_time"] - r["start_time"]) / 3600
        share = r.get("overlap_share")
        for name in (r["channel"], "all"):
            c = out.setdefault(
                name,
                {
                    "hours": 0.0,
                    "unmeasured_hours": 0.0,
                    "dropped_hours": dict.fromkeys(thresholds, 0.0),
                },
            )
            c["hours"] += hours
            if share is None:
                c["unmeasured_hours"] += hours
                continue
            for threshold in thresholds:
                if share > threshold:
                    c["dropped_hours"][threshold] += hours
    return dict(sorted(out.items(), key=lambda kv: (kv[0] == "all", kv[0])))


# --- stage 2 of a student: pseudo-labels and human labels in one mixture (D106) ------------------


def mixture_weights(
    groups: Sequence[str | None], hours: Sequence[float], human_share: float
) -> list[float]:
    """Each clip's chance of being drawn, summing to 1.

    ``groups[i]`` is clip i's channel for a pseudo-labelled clip and ``None`` for a human-labelled
    one; ``hours[i]`` is its length. The human labels take ``human_share`` of the draws. The rest
    is split between channels in proportion to the *square root* of their hours (roadmap §B,
    2026-09-26: no channel is capped, and a 22 h channel weighs about 2x a 5 h one, not 4.4x).
    Inside a channel, and inside the human labels, a clip is drawn in proportion to its length.
    With no pseudo-labelled clip, or no human one, the other kind takes every draw.
    """
    if len(groups) != len(hours):
        raise ValueError("groups and hours must describe the same clips")
    if not 0.0 < human_share < 1.0:
        raise ValueError(f"human_share must be in (0, 1), not {human_share}")
    total: dict[str | None, float] = {}
    for g, h in zip(groups, hours, strict=True):
        if h <= 0:
            raise ValueError("every clip needs a positive length")
        total[g] = total.get(g, 0.0) + h
    channels = {g: h for g, h in total.items() if g is not None}
    roots = sum(h**0.5 for h in channels.values())
    share: dict[str | None, float] = {}
    if None in total:
        share[None] = human_share if channels else 1.0
    for g, h in channels.items():
        share[g] = (1.0 - human_share if None in total else 1.0) * h**0.5 / roots
    return [share[g] * h / total[g] for g, h in zip(groups, hours, strict=True)]


def draw_epoch(weights: Sequence[float], n: int, seed: int) -> list[int]:
    """The ``n`` clips of one epoch, as indices drawn with replacement by ``weights``; repeatable
    from ``seed``, so a resumed run trains the epochs it would have."""
    import random

    return random.Random(seed).choices(range(len(weights)), weights=list(weights), k=n)

In [ ]:
import gc
import math
import warnings

import numpy as np
import torch
import torch.utils.checkpoint
import transformers
from huggingface_hub import HfApi, hf_hub_download, snapshot_download

import evalkit
import ftkit
import sweep

# transformers' generate() warns on every call (max_new_tokens vs max_length); it is noise here and
# buries the training log. Errors still show.
transformers.logging.set_verbosity_error()
warnings.filterwarnings("ignore", module="transformers")
ftkit.fast_cuda()
TOKEN = os.environ["HF_TOKEN"]
api = HfApi(token=TOKEN)
api.create_repo(OUT_REPO, repo_type="model", private=True, exist_ok=True)

DATA = ftkit.download_dataset(DATA_LOCAL)  # labels and the scorer; the audio follows
export = json.loads((DATA / "training" / "manifest.json").read_text())
evalkit.check_export(export, DATASET_EXPORT)
splits = ftkit.load_splits(DATA)
if SMOKE:
    splits = {"train": splits["train"][::50], "val": splits["val"][::30], "gold": splits["gold"][::16]}
AUDIO_SPLITS = ('val', 'gold')  # the splits this notebook decodes or trains on
episodes = [r["episode_id"] for name in AUDIO_SPLITS for r in splits[name]]
with ftkit.timed(f"downloading {len(set(episodes))} recordings"):
    DATA = ftkit.download_dataset(DATA_LOCAL, episodes=episodes, analytics=False)
with ftkit.timed("reading them into RAM"):
    store = ftkit.AudioStore(DATA, episodes)
score = ftkit.harness_scorer(DATA, FT)
print({k: len(v) for k, v in splits.items()}, f"| audio in RAM: {store.gib:.1f} GiB | export",
      export["exported_at"][:10], "|", score.fold_version)

FLEX_DIR = snapshot_download(MODEL_ID)
sys.path.insert(0, FLEX_DIR)
from indic_transcribe import MODES, IndicTranscribe  # noqa: E402


def load_model(path):
    """Weights from `path` on the GPU in fp32, as `model` (which the decoder and the loss read)."""
    global asr, model, featurize
    asr = IndicTranscribe.from_pretrained(str(path), device="cuda", dtype=torch.float32)
    model, featurize = asr.model, asr.fe
    if globals().get("GRAD_CKPT"):
        for layer in model.model.encoder.layers:
            layer._forward = layer.forward
            layer.forward = lambda *a, _l=layer, **k: torch.utils.checkpoint.checkpoint(_l._forward, *a, use_reentrant=False, **k)
    return model


def free_model():
    for name in ("optimizer", "model", "asr"):
        globals().pop(name, None)
    gc.collect()
    torch.cuda.empty_cache()


load_model(FLEX_DIR)
tk = asr.tokenizer
itn, romanized = MODES[MODE]
PROMPT = tk.encode_prompt(LANG, itn=itn, romanized=romanized)
EOS, PAD, OFFSET = tk.eos_id, tk.pad_id, tk.spl_size
UNK_MULTI = tk.multi.unk_id()

In [ ]:
import shutil

DEV_DIGITS = str.maketrans("०१२३४५६७८९", "0123456789")


def normalize(text: str) -> str:
    """Map the characters the tokenizer lacks onto ones fold.py scores as identical."""
    return (text.replace("।", ".").translate(DEV_DIGITS).replace("—", "-")
            .replace("‍", "").replace("‌", ""))


def features(wav: torch.Tensor, lens: torch.Tensor):
    feats, flens = featurize(wav, lens)  # fp32 even inside autocast: it disables autocast itself
    mask = (torch.arange(feats.size(2), device=feats.device)[None, :] < flens[:, None]).long()
    return feats, flens, mask


def transcribe(rows, **gen):
    clips = [store.clip_f32(r) for r in rows]
    wav = torch.zeros(len(rows), ftkit.pad_len(max(len(c) for c in clips), PAD_TO_S))
    for i, c in enumerate(clips):
        wav[i, :len(c)] = c
    feats, _, mask = features(wav.cuda(), torch.tensor([len(c) for c in clips], device="cuda"))
    prompt = torch.tensor([PROMPT] * len(rows), device="cuda")
    with torch.autocast("cuda", dtype=torch.bfloat16):
        out = model.generate(input_features=feats, attention_mask=mask, decoder_input_ids=prompt,
                             do_sample=False, num_beams=1, eos_token_id=EOS, pad_token_id=PAD,
                             **{"max_new_tokens": 300, **gen})
    return [tk.decode(tk.strip_prompt_and_trim(o.tolist(), PROMPT)) for o in out]


# Loop retry, fixed before scoring anything with it: greedy decoding occasionally sticks on a filler
# ("अँ. अँ. अँ.") to the length limit. A global repetition ban would also hit real repeats the
# references keep ("कोही छैन। कोही छैन"), so only a clip whose greedy output loops is decoded
# again, alone, with a repetition penalty, no repeated 6-token phrase, and a length cap from its
# duration (the densest training label is 12.6 tokens/s).
MAX_TOKENS_PER_S = 13.0
RETRY = {"repetition_penalty": 1.2, "no_repeat_ngram_size": 6}


def retry_one(row):
    cap = min(300, math.ceil(MAX_TOKENS_PER_S * ftkit.duration(row)))
    return transcribe([row], max_new_tokens=cap, **RETRY)[0]


def decode(rows):
    """The decoder every score in 03a-03e comes from: greedy, batched, plus the loop retry.
    Returns the texts, each clip's compute seconds and the retries."""
    model.eval()
    d = ftkit.RetryLoops(transcribe, retry_one)
    with torch.no_grad():
        texts, compute = ftkit.transcribe_rows(rows, d, budget_s=EVAL_BUDGET_S, max_items=EVAL_ITEMS,
                                               pad_to_s=PAD_TO_S)
    return texts, compute, d.log


def save_best(model):
    """`model` as `OUT/best/`, in bf16, with the code and tokenizer files IndicTranscribe loads."""
    dst = OUT / "best"
    model.save_pretrained(dst, state_dict={k: v.to(torch.bfloat16) for k, v in model.state_dict().items()})
    for f in Path(FLEX_DIR).iterdir():
        if f.suffix in {".py", ".model", ".md"} or f.name in {"tokenizer_config.json", "generation_config.json",
                                                             "feature_extractor.safetensors"}:
            shutil.copy(f, dst / f.name)


def card(description, **extra):
    """The model card's fields that do not depend on the scores: what the model is, and which
    export, fold and code the run used."""
    return {
        "name": MODEL_NAME,
        "description": description,
        "architecture": "Canary-style enc-dec: 32L conformer + 24L transformer decoder, 1.2B",
        "base_model": MODEL_ID,
        "decoder": "greedy+retry",
        "notebook": NOTEBOOK,
        "dataset_export": export["exported_at"],
        "train_export": {k: export.get(k) for k in ("exported_at", "git_commit", "row_count",
                                                    "normalization_version", "label_version")},
        "fold_version": score.fold_version,
        "kits": evalkit.kit_digests(FT),
        **extra,
    }


def fetch_json(remote):
    """A JSON file of OUT_REPO, or None when it is not there."""
    if not api.file_exists(OUT_REPO, remote):
        return None
    return json.loads(Path(hf_hub_download(OUT_REPO, remote, token=TOKEN)).read_text("utf-8"))


def uploaded(run):
    """A run's result row: OUT_REPO has it once the run is trained, scored and uploaded."""
    return fetch_json(f"{RUN_PREFIX}/{run}/result.json")


def run_counts(run):
    """A run's per-clip counts on gold and val, for another run to be paired against."""
    return fetch_json(f"{RUN_PREFIX}/{run}/per_clip.json")


def upload_run(out, run, message, weights):
    with ftkit.timed(f"uploading {run} ({'with' if weights else 'without'} weights)"):
        api.upload_folder(repo_id=OUT_REPO, folder_path=str(out), path_in_repo=f"{RUN_PREFIX}/{run}",
                          ignore_patterns=None if weights else ["best/*"], commit_message=f"{run}: {message}")


def download_weights(run):
    """`<run>/best/` of OUT_REPO as a local folder `load_model` reads."""
    with ftkit.timed(f"downloading the weights of {run}"):
        root = snapshot_download(OUT_REPO, allow_patterns=[f"{RUN_PREFIX}/{run}/best/*"], token=TOKEN)
    best = Path(root) / RUN_PREFIX / run / "best"
    if not (best / "model.safetensors").exists():
        raise FileNotFoundError(f"{OUT_REPO} holds no weights for {RUN_PREFIX}/{run}")
    return best


def line(m):
    return f"{m['wer']:6.2f} ({ftkit.sid(m)})"

In [ ]:
SETS = tuple(evalkit.BENCHMARKS)  # every public set; name fewer to skip some
LIMIT = 40 if SMOKE else None     # clips per set
# Each set's crosstalk and SNR, measured once by scripts/measure_benchmark_overlap.py
# (WER-Breakdown.md).
CONDITIONS = evalkit.fetch_conditions(OUT_REPO, TOKEN, FT / "conditions", SETS)


def done_sets(run):
    """The public sets OUT_REPO already holds a summary of for this run."""
    prefix = f"{RUN_PREFIX}/{run}/benchmarks/"
    return {Path(f).stem for f in api.list_repo_files(OUT_REPO)
            if f.startswith(prefix) and f.endswith(".json") and Path(f).stem in evalkit.BENCHMARKS}


def score_sets(run):
    """Decode the public sets with the weights in `model` and upload each as it finishes."""
    out = OUT_ROOT / run
    done = done_sets(run)
    for name in SETS:
        if name in done:
            print(f"{run} {name}: already in {OUT_REPO}, skipped")
            continue
        evalkit.run_benchmarks(out, decode=decode, score=score, work=FT / "benchmarks", token=TOKEN,
                               names=[name], limit=LIMIT, run=run, conditions_dir=CONDITIONS)
        api.upload_folder(repo_id=OUT_REPO, folder_path=str(out), path_in_repo=f"{RUN_PREFIX}/{run}",
                          allow_patterns=[f"benchmarks/{name}.json*", f"harness/errors/{name}.parquet"],
                          commit_message=f"{run}: {name}")


def set_files(run, name):
    """(summary, per-clip lines) of a run on a public set, from OUT_REPO; None when it has none."""
    remote = f"{RUN_PREFIX}/{run}/benchmarks/{name}.json"
    if not api.file_exists(OUT_REPO, remote):
        return None
    lines = Path(hf_hub_download(OUT_REPO, remote + "l", token=TOKEN)).read_text("utf-8").splitlines()
    return fetch_json(remote), [json.loads(x) for x in lines]


def public_table(runs, against="base"):
    """Every run on every public set, and each run minus `against` with the set's group resampled."""
    table = {}
    for name in SETS:
        files = {run: set_files(run, name) for run in runs}
        ref = files.get(against)
        print(f"\n{name}: {evalkit.BENCHMARKS[name].what}")
        for run, got in files.items():
            if got is None:
                print(f"  {run:<14} not scored")
                continue
            m, lines = got
            entry = {k: m[k] for k in ("wer", "raw_wer", "plain_wer", "sub", "del", "ins", "clips")}
            entry["by"] = {k: v["wer"] for k, v in (m.get("by") or {}).items()}
            text = f"  {run:<14} {line(m)}  plain {m['plain_wer']:.2f}"
            if ref is not None and run != against:
                entry["vs"] = evalkit.pair_benchmark(ref[1], lines)
                d, lo, hi = entry["vs"]["all"]
                text += f"  | minus {against} {d:+.2f} [{lo:+.2f}, {hi:+.2f}]"
            table.setdefault(name, {})[run] = entry
            print(text)
    means = {run: float(np.mean([table[n][run]["wer"] for n in SETS]))
             for run in runs if all(run in table.get(n, {}) for n in SETS)}
    print("\npublic mean, folded WER:", "  ".join(f"{run} {v:.2f}" for run, v in means.items()))
    return table, means

## The two models, and the blend

In [ ]:
from safetensors.torch import load_file


def blend(alpha, tuned):
    """`model` = (1 - alpha) * base + alpha * fine-tuned (WiSE-FT), in place on fresh base weights.

    `tuned` is the fine-tune's state dict. Every float tensor it holds is interpolated with the
    same alpha; a buffer that is not float is taken from the fine-tune. A tensor two names share
    (the tied output head) is blended once. Returns how many tensors were blended."""
    load_model(FLEX_DIR)
    seen, n = set(), 0
    with torch.no_grad():
        for key, w in model.state_dict().items():
            if key not in tuned or w.data_ptr() in seen:
                continue
            seen.add(w.data_ptr())
            t = tuned[key].to(w.device)
            if w.is_floating_point():
                w.lerp_(t.to(w.dtype), alpha)
                n += 1
            else:
                w.copy_(t)
    return n

In [ ]:
source = SOURCE or (fetch_json(f"{RUN_PREFIX}/ablation.json") or {}).get("winner")
if not source:
    raise RuntimeError(f"no ablation.json under {OUT_REPO}/{RUN_PREFIX}: run 03c, or set SOURCE in Config")
tuned_row = uploaded(source)
if tuned_row is None:
    raise RuntimeError(f"{source} has no result in {OUT_REPO}/{RUN_PREFIX}")
tuned = load_file(str(download_weights(source) / "model.safetensors"))  # bf16, on the CPU
base_row, base_counts, tuned_counts = uploaded("base"), run_counts("base"), run_counts(source)
if base_row is None:
    print("03b has not scored base on gold and val: the blends will not be paired against it")
print(f"blending base with {source}: val {line(tuned_row['val'])} | gold {line(tuned_row['gold'])}")

# The recipe's check: at alpha = 1 the blend is the fine-tune, tensor for tensor.
n = blend(1.0, tuned)
state = model.state_dict()
wrong = [k for k, t in tuned.items() if k in state and not torch.equal(state[k].cpu(), t.to(state[k].dtype))]
assert not wrong, f"alpha = 1 does not reproduce the fine-tune: {wrong[:5]}"
print(f"{n} float tensors blended; alpha = 1 reproduces {source} exactly")
free_model()


def blend_run(alpha):
    # Score one blend on gold, val and the public sets, and upload it (no weights).
    global OUT
    run = f"blend-{round(alpha * 100):03d}"
    row = uploaded(run)
    if row is not None and done_sets(run) >= set(SETS):
        print(f"{run}: already in {OUT_REPO}, skipped")
        return
    OUT = OUT_ROOT / run
    OUT.mkdir(parents=True, exist_ok=True)
    blend(alpha, tuned)
    try:
        if row is None:
            references = {k: v for k, v in (("base", base_counts), (source, tuned_counts)) if v}
            evalkit.evaluate_run(OUT, run, splits=splits, decode=decode, score=score,
                                 card=card(f"{1 - alpha:g} x base + {alpha:g} x {source} (WiSE-FT)",
                                           name=f"{MODEL_NAME} blend {alpha:g}"),
                                 meta={"recipe": "blend", "alpha": alpha, "source": source,
                                       "seed": tuned_row.get("seed")},
                                 references=references)
            upload_run(OUT, run, f"blend of base and {source}", weights=False)
        score_sets(run)
    finally:
        free_model()

## One blend per cell

Gold and val, then the public sets, each uploaded as it finishes.

In [ ]:
blend_run(ALPHAS[0])

In [ ]:
blend_run(ALPHAS[1])

In [ ]:
blend_run(ALPHAS[2])

## The fine-tune on the public sets

03b scores it only when it is named in its `RUNS`. If it was not, it is done here, so the report
has the alpha = 1 row.

In [ ]:
if done_sets(source) >= set(SETS):
    print(f"{source}: every public set already in {OUT_REPO}")
else:
    load_model(download_weights(source)).eval()
    try:
        score_sets(source)
    finally:
        free_model()

## Report, and the choice

Base, each blend and the fine-tune on val, gold and the public sets, then the blend the rule
keeps. `blend.json` names it for 03e.

In [ ]:
runs = ["base", *[f"blend-{round(a * 100):03d}" for a in ALPHAS], source]
table, means = public_table(runs)
rows = [{**tuned_row, "alpha": 1.0}]
for alpha in ALPHAS:
    if (row := uploaded(f"blend-{round(alpha * 100):03d}")) is not None:
        rows.append(row)
winner, why = sweep.choose_blend(rows, tolerance=TOLERANCE)
print(f"\n{'model':<16}{'alpha':>6}  {'val':<34}{'gold':<34}{'public mean':>12}")
for r in ([{**base_row, "alpha": 0.0}] if base_row else []) + sorted(rows, key=lambda r: r["alpha"]):
    mean = means.get(r["run_name"])
    tag = " <- chosen" if r is winner else ""
    print(f"{r['run_name']:<16}{r['alpha']:>6g}  {line(r['val']):<34}{line(r['gold']):<34}"
          f"{'-' if mean is None else format(mean, '.2f'):>12}{tag}")
for r in rows:
    for split in ("val", "gold"):
        for name, vs in r[split]["vs"].items():
            d, lo, hi = vs["all"]
            print(f"{r['run_name']} {split} minus {name}: {d:+.2f} [{lo:+.2f}, {hi:+.2f}]")
print("\nchosen:", winner["run_name"], "|", why)
summary = {"rule": f"val WER; the blend closest to base within {TOLERANCE:g} points of the fine-tune (2026-09-27)",
           "tolerance": TOLERANCE, "source": source, "winner": winner["run_name"], "alpha": winner["alpha"],
           "why": why, "rows": rows, "public": table, "public_mean": means}
(OUT_ROOT / "blend.json").write_text(json.dumps(summary, indent=1, ensure_ascii=False))
api.upload_file(path_or_fileobj=str(OUT_ROOT / "blend.json"), path_in_repo=f"{RUN_PREFIX}/blend.json",
                repo_id=OUT_REPO, commit_message=f"{RUN_PREFIX}: blend, chosen {winner['run_name']}")